In [1]:
# Packages not preinstalled on Kaggle that these scripts need.
# GalaxyMNIST is pinned to the commit scripts/train.py's original comments reference,
# for exact reproducibility.
!pip install -q einops torchinfo h5py pygame
!pip install -q "git+https://github.com/mwalmsley/galaxy_mnist.git@c1fe9853a00bc34b2ff082585c6bb1654d34d239"


  Preparing metadata (setup.py) ... done


In [2]:
import os, sys
from pathlib import Path

SMOKE_TEST = False  # True -> overrides EPOCHS to 2 in every script, for a fast pipeline check

REPO_ROOT = Path("/kaggle/working/s4d_repo")
(REPO_ROOT / "model").mkdir(parents=True, exist_ok=True)
(REPO_ROOT / "scripts").mkdir(parents=True, exist_ok=True)
(REPO_ROOT / "logs").mkdir(parents=True, exist_ok=True)

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("Working directory:", os.getcwd())


Working directory: /kaggle/working/s4d_repo


In [3]:
%%writefile utils.py
"""
Minimal stand-in for `utils.set_pbar_style`, imported by `scripts/train.py`.

`scripts/train.py` does `from utils import set_pbar_style`, but no `utils.py`
was present in the uploaded s4d.zip. Without this stub, importing
`train.py` -- which `train_hybrid.py`, `train_cnn_only.py`, and
`train_hybrid_scale.py` all do, to reuse its `train()` function -- fails
with `ModuleNotFoundError: No module named 'utils'`.

`set_pbar_style` only ever affected tqdm progress-bar colors in the
original notebook-style script body; it has no effect on training logic,
so this no-op-safe stub is a purely cosmetic substitute.
"""


def set_pbar_style(bar_fill_color="#FFFFFF", text_color="#FFFFFF"):
    """Cosmetic no-op. The original styling implementation wasn't in the
    uploaded zip, so this stub just accepts the same call signature used
    in scripts/train.py without changing tqdm's behavior."""
    return None


Writing utils.py


In [4]:
%%writefile kaggle_extras.py
"""
kaggle_extras.py -- not part of the original s4d.zip.

Shared helpers added for the Kaggle notebook run:
  - a torchinfo model summary for every model trained (architecture +
    per-layer param counts), printed and saved alongside each run's
    other artifacts;
  - the same classification metrics the LaTeX report uses throughout its
    master results table (accuracy, precision/recall/F1 macro, one-vs-rest
    ROC-AUC macro), for the two scripts (train_hybrid.py,
    train_hybrid_scale.py) that didn't already compute them;
  - staging of weights, training curves, and confusion matrices into a
    flat, easy-to-find/download location directly under /kaggle/working/,
    instead of nested inside this repo's own working directory.
"""
import glob
import os
import re
import shutil

import numpy as np
import torch
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
from sklearn.preprocessing import label_binarize

KAGGLE_ROOT = "/kaggle/working"
OUTPUTS_ROOT = os.path.join(KAGGLE_ROOT, "outputs")


def safe_slug(name):
    """Turn a human-readable model name into a filesystem-safe slug."""
    s = re.sub(r"[^\w\-.]+", "_", name.strip())
    return re.sub(r"_+", "_", s).strip("_")


def output_dir(script_tag):
    """/kaggle/working/outputs/<script_tag>/ -- created on first use.

    Falls back to a local ./outputs/<script_tag>/ if /kaggle/working isn't
    writable (e.g. testing outside Kaggle), so a script never crashes
    purely because it isn't running in a Kaggle kernel.
    """
    d = os.path.join(OUTPUTS_ROOT, script_tag)
    try:
        os.makedirs(d, exist_ok=True)
    except OSError:
        d = os.path.join("outputs", script_tag)
        os.makedirs(d, exist_ok=True)
    return d


def print_and_save_summary(model, input_size, name, out_dir):
    """Run torchinfo.summary(), print it, and save the text alongside the
    other artifacts for this run. Falls back to a plain parameter count if
    torchinfo isn't importable for some reason, rather than crashing a
    training run over a reporting nicety."""
    header = f"\n{'=' * 70}\nModel summary: {name}\n{'=' * 70}"
    print(header)
    try:
        from torchinfo import summary as _summary
        stats = _summary(model, input_size=input_size, verbose=0,
                          col_names=("input_size", "output_size", "num_params"))
        text = str(stats)
    except Exception as exc:  # pragma: no cover
        n_params = sum(p.numel() for p in model.parameters())
        text = f"(torchinfo summary unavailable: {exc})\nTotal params: {n_params:,}"
    print(text)
    with open(os.path.join(out_dir, f"{safe_slug(name)}_summary.txt"), "w") as f:
        f.write(header + "\n" + text + "\n")
    return text


def save_weights(model, name, out_dir):
    """Save model.state_dict() so trained weights survive past the Kaggle
    session, not just the in-memory results dict."""
    path = os.path.join(out_dir, f"{safe_slug(name)}_weights.pth")
    torch.save(model.state_dict(), path)
    print(f"Saved weights -> {path}")
    return path


def compute_classification_metrics(all_targets, all_preds, all_probs, class_names):
    """Precision/recall/F1 (macro) + one-vs-rest ROC-AUC (macro) -- the
    same metrics used throughout the LaTeX report's master results table
    (Acc / F1 / Prec / Rec / AUC columns)."""
    labels_arr = list(range(len(class_names)))
    prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
        all_targets, all_preds, labels=labels_arr, average="macro", zero_division=0
    )
    y_true_bin = label_binarize(all_targets, classes=labels_arr)
    probs_arr = np.array(all_probs)
    try:
        auc_macro = float(roc_auc_score(y_true_bin, probs_arr, average="macro", multi_class="ovr"))
    except ValueError:
        # can happen if a class is entirely absent from the test batch
        auc_macro = float("nan")
    return {
        "precision_macro": float(prec_macro),
        "recall_macro": float(rec_macro),
        "f1_macro": float(f1_macro),
        "roc_auc_macro": auc_macro,
    }


def stage_outputs(out_dir, *patterns):
    """Copy every file matching each glob pattern into out_dir. Copies,
    not moves -- originals stay where the script originally wrote them
    too, in case anything else in this notebook still reads them from
    their original relative path (the results-display cells do)."""
    copied = []
    for pattern in patterns:
        for src in glob.glob(pattern):
            dst = os.path.join(out_dir, os.path.basename(src))
            shutil.copy2(src, dst)
            copied.append(dst)
    print(f"Staged {len(copied)} file(s) -> {out_dir}")
    return copied


Writing kaggle_extras.py


In [5]:
%%writefile model/__init__.py
"""
Galaxy Classification Model Package
"""

from .gclassifier import GalaxyClassifierS4D
from .gclassifier_hybrid import GalaxyClassifierCNNS4D
from .gclassifier_cnn_only import GalaxyClassifierCNNOnly
from .cnn_stem import CNNStem
from . import functions
from .interface import ModelInterface

# NOTE (Kaggle notebook): the interactive pygame GUI is unrelated to
# training and isn't used by any of the four training scripts this
# notebook runs. The import is wrapped defensively so that an unrelated
# GUI/display dependency hiccup in a headless Kaggle kernel can never
# block model training (pygame does import cleanly headless in testing,
# so this is a belt-and-suspenders guard, not a workaround for a known
# failure).
try:
    from .gui import GalaxyExplorerGUI
except Exception as _gui_exc:  # pragma: no cover
    GalaxyExplorerGUI = None
    print(f"[model/__init__] GalaxyExplorerGUI unavailable ({_gui_exc}); "
          f"not needed for training, continuing.")

__all__ = [
    'GalaxyClassifierS4D',
    'GalaxyClassifierCNNS4D',
    'GalaxyClassifierCNNOnly',
    'CNNStem',
    'functions',
    'ModelInterface',
    'GalaxyExplorerGUI',
]


Writing model/__init__.py


In [6]:
%%writefile model/tlts.py
import torch
import torch.nn as nn

class TakeLastTimestep(nn.Module):
    """
    Module that extracts the last timestep from a sequence.

    This layer is used to summarize sequence outputs from recurrent 
    or sequence models by taking only the final timestep as a feature vector.

    Parameters
    ----------
    None

    Input
    -----
    x : torch.Tensor
        Input tensor of shape (B, L, D), where
        B : batch size,
        L : sequence length,
        D : feature dimension.

    Returns
    -------
    out : torch.Tensor
        Output tensor of shape (B, D), corresponding to the last timestep
        of each sequence in the batch.
    """
    def forward(self, x):
        # x: (B, L, D)
        # FIX (Kaggle notebook, applied on top of the uploaded zip): this
        # method was `return x.mean(dim=1)` -- mean-pooling, not
        # last-timestep pooling. That contradicted this class's own name
        # and docstring, its own commented-out self-test below (which
        # asserts the output equals x[:, -1, :]), and the reference
        # `TakeLastTimestep` in notebook-best-s4d-model_1_.ipynb, which
        # correctly implements last-timestep pooling. Every S4D model in
        # this repo (GalaxyClassifierS4D, GalaxyClassifierCNNS4D, and the
        # grid model in train_ablation.py) uses this class unconditionally,
        # so as shipped every "last pooling" run was silently mean-pooling.
        # Restored to genuine last-timestep pooling to match both the
        # documented intent and the reference notebook.
        return x[:, -1, :]

"""
if __name__ == "__main__":
    print("Testing tlts code")

    layer = TakeLastTimestep()
    print("Layer established")

    x = torch.randn(3, 6, 2)
    print(f"Input shape: {x.shape}")

    output = layer(x)
    print(f"Output shape: {output.shape}")
    print(f"Expected (3, 2): {output.shape == (3, 2)}")
    print(f"Match? {torch.allclose(x[0, -1, :], output[0, :])}")

    print("Successful")
"""
"""
Explaination:
The TakeLastTimeStep layer transforms an input tensor of shape (B, L, D) 
into an output tensor of shape (B, D) by indexing the last position.
The hidden state at position L has been updated by all L previous inputs
and by the time model reaches position L-1, which is the last time step, 
the state has came across and collected information from all inputs u(0) 
through u(L-1) and therefore the (B, D) tensor at the final position
serves as a compressed summary of the entire (B, L, D) sequence.
This is identical to how RNNs, LSTMs, and GRUs use their final hidden state
for classification tasks — the last timestep naturally accumulates the 
history of the whole sequence through the recurrent processes.
"""


Writing model/tlts.py


In [7]:
%%writefile model/hilbert.py
import torch   
import torch.nn as nn


class HilbertScan(nn.Module):
    """
    Reorders pixels according to a Hilbert Curve for multi-channel images.
    
    The Hilbert curve is a space-filling curve that preserves spatial locality
    when mapping 2D coordinates to 1D sequences. This module applies the same
    Hilbert curve pattern to each channel independently, then reorganizes the
    output so the sequence dimension comes first.
    
    Supports grayscale (C=1) or RGB (C=3) images.
    
    Attributes
    ----------
    indices : torch.LongTensor
        Precomputed Hilbert curve indices for an n×n grid, stored as a
        non-trainable buffer.
    
    Input
    -----
    x : torch.Tensor
        Input tensor of shape (B, C, H, W), where
        B : batch size
        C : number of channels
        H : height (n)
        W : width (n)
    
    Returns
    -------
    out : torch.Tensor
        Reordered tensor of shape (B, seq_len, C) where seq_len = H*W = n*n.
        Pixels are arranged according to the Hilbert curve traversal order.
    """
    def __init__(self, n=64):
        """Initialize HilbertScan with precomputed indices for an n x n grid.

        Parameters
        ----------
        n : int, optional
            Grid size (must be a power of 2). Default 64, which keeps the
            existing GalaxyClassifierS4D baseline (scanning the raw 64x64
            image) unaffected. The hybrid CNN+S4D classifier passes a
            smaller n (e.g. 16) since it scans the CNN stem's downsampled
            feature map instead of the raw image.
        """
        super().__init__()
        self.n = n
        indices = self.get_hilbert_indices(n)
        self.register_buffer('indices', indices)

    def _rot(self, s, x, y, rx, ry):
    
        if ry == 0:                  # Bottom half of the current square
            if rx == 1:              # Bottom-right quadrant
                x = s - 1 - x        # Reflect over diagonal
                y = s - 1 - y
            x, y = y, x              # Swap x and y for 90° rotation
        return x, y


    def _d2xy(self, n, d):
        """
        Convert 1D Hilbert curve distance to 2D coordinates.
        
        This implements the Hilbert curve mapping algorithm that converts
        a linear distance along the curve to (x, y) coordinates.
        
        Parameters
        ----------
        n : int
            Size of the grid (must be a power of 2).
        d : int
            Distance along the Hilbert curve (0 to n²-1).
        
        Returns
        -------
        tuple of int
            (x, y) coordinates in the grid.
        """
        x = 0
        y = 0
        t = d
        s = 1
#Determine which quadrant of the current square this distance is in
        while s < n:
            rx = (t // 2) & 1
            ry = (t ^ rx) & 1
#Rotate and/or reflect coordinates depending on quadrant
            x, y = self._rot(s, x, y, rx, ry)
            x += s * rx
            y += s * ry
#Move to next level of recursion (divide distance by 4 for next smaller square)
            t //= 4
            s *= 2

        return x, y

    def get_hilbert_indices(self, n):
        """
        Generate Hilbert curve indices for an n x n grid.
        
        Creates a lookup table that maps Hilbert curve positions to
        flattened array indices for a 2D grid.
        
        Parameters
        ----------
        n : int
            Grid size (must be a power of 2).
        
        Returns
        -------
        torch.LongTensor
            Tensor of shape (n²,) containing flattened indices following
            the Hilbert curve traversal order.
        """
        indices = []
        for d in range(n * n):
            x, y = self._d2xy(n, d)
            # GalaxyMNIST is 64x64, power of 2
            if x < n and y < n:
                indices.append(y * n + x)
        return torch.LongTensor(indices)

    def forward(self, x):
        """
        Apply Hilbert curve reordering to input images.
        
        Parameters
        ----------
        x : torch.Tensor
            Input images of shape (B, C, H, W).
        
        Returns
        -------
        torch.Tensor
            Reordered tensor of shape (B, seq_len, C) where seq_len = H*W,
            with pixels arranged in Hilbert curve order.
        """
        # x: (B, C, H, W)
        B, C, H, W = x.shape
        x = x.view(B, C, -1)           # Flatten each channel: (B, C, H*W)
        x = x[:, :, self.indices]      # Reorder according to Hilbert: (B, C, H*W)
        x = x.permute(0, 2, 1)         # (B, seq_len, C) so sequence dimension is 1D
        return x
if __name__ == "__main__":
    import torch

    img = torch.arange(64*64).view(1,1,64,64).float()
    hilbert = HilbertScan()
    out = hilbert(img)

    print(out[0, :20, 0])

Writing model/hilbert.py


In [8]:
%%writefile model/cnn_stem.py
import torch
import torch.nn as nn


class CNNStem(nn.Module):
    """
    Convolutional stem for the CNN-stem -> S4D hybrid classifier.

    RESEARCH VERSION (post-course). The course version of this stem was
    constrained to plain Conv+GELU (no BatchNorm/LayerNorm) for eventual
    bare-metal RISC-V portability. That constraint is dropped here since
    we're now optimizing purely for accuracy -- if a bare-metal export is
    ever needed again, GroupNorm's running stats can be folded into the
    preceding conv's weights at export time (standard conv-BN/GN fusion),
    so this doesn't have to be a permanent trade-off even for that goal.

    Key differences from the course version:
      1. A stride-1 "detail" conv runs FIRST, at full input resolution,
         before any downsampling happens. The old stem's first conv was
         already stride-2, so it only ever saw a raw 3x3 window of
         un-processed pixels before halving resolution -- thin, low-
         contrast structures (e.g. dust lanes in edge-on spirals, which
         is exactly the signal needed to separate Smooth Cigar from
         Edge-on Disk) had no chance to be extracted before being pooled
         away. Now there's a full-res feature-extraction pass first.
      2. GroupNorm after every conv (stable training, no batch-size
         dependence, no running stats to worry about -- unlike
         BatchNorm, GroupNorm's stats are computed per-sample so it
         behaves identically in train/eval).
      3. More channel capacity (mid_channels default raised 16 -> 32).
      4. Residual add on the stride-1 block (cheap, helps optimization,
         doesn't change spatial dims so it's a free add).

    Two variants, selected via `reduction` (same semantics as before):
      - reduction=16: three conv stages, 64x64 -> 64x64 (stride1) ->
        32x32 -> 16x16   => 16x sequence-length cut (4096->256)
      - reduction=4:  64x64 -> 64x64 (stride1) -> 32x32
                                                    => 4x cut (4096->1024)

    Parameters
    ----------
    in_channels : int
        Number of input image channels (1 grayscale, 3 RGB). Use 3 --
        color carries the dust-lane / reddening signal that grayscale
        (channel-averaged) input throws away.
    d_model : int, optional
        Output channel count of the stem, feeding S4D's d_model. Default 64.
    mid_channels : int, optional
        Hidden channel width of the stem's early conv stages. Default 32
        (was 16 in the course version -- more capacity now that accuracy,
        not param-count / embedded footprint, is the objective).
    reduction : int, optional
        Spatial / sequence-length reduction factor. One of {4, 16}.
        Default 16.
    dropout : float, optional
        Spatial dropout (Dropout2d) applied after the stride-1 block, as
        light regularization for the ~8k-image dataset. Default 0.1.

    Input
    -----
    x : torch.Tensor, shape (B, in_channels, 64, 64)

    Returns
    -------
    torch.Tensor
        shape (B, d_model, 16, 16) if reduction=16,
        shape (B, d_model, 32, 32) if reduction=4.
    """

    def __init__(self, in_channels, d_model=64, mid_channels=32, reduction=16, dropout=0.1):
        super().__init__()
        if reduction not in (4, 16):
            raise ValueError(f"reduction must be 4 or 16, got {reduction}")
        self.reduction = reduction

        def gn(channels):
            # GroupNorm needs num_groups | channels; 8 groups is a safe
            # default for the channel counts used here (32, 64).
            groups = 8 if channels % 8 == 0 else 1
            return nn.GroupNorm(groups, channels)

        # --- Stage 0: full-resolution detail extraction (stride 1) ---
        # This is the change that matters most: features are computed at
        # the input's native 64x64 resolution before anything is thrown
        # away, so thin/low-contrast structures (dust lanes, arm edges)
        # actually get a chance to be represented.
        self.stem_conv = nn.Conv2d(in_channels, mid_channels, kernel_size=3, stride=1, padding=1)
        self.stem_norm = gn(mid_channels)
        self.stem_act = nn.GELU()

        self.res_conv = nn.Conv2d(mid_channels, mid_channels, kernel_size=3, stride=1, padding=1)
        self.res_norm = gn(mid_channels)
        self.res_act = nn.GELU()
        self.drop = nn.Dropout2d(dropout)

        # --- Downsampling stages ---
        if reduction == 16:
            # 64 -> 32 -> 16
            self.down1 = nn.Conv2d(mid_channels, mid_channels, kernel_size=3, stride=2, padding=1)
            self.down1_norm = gn(mid_channels)
            self.down1_act = nn.GELU()

            self.down2 = nn.Conv2d(mid_channels, d_model, kernel_size=3, stride=2, padding=1)
            self.down2_norm = gn(d_model)
            self.down2_act = nn.GELU()
        else:
            # 64 -> 32
            self.down1 = nn.Conv2d(mid_channels, d_model, kernel_size=3, stride=2, padding=1)
            self.down1_norm = gn(d_model)
            self.down1_act = nn.GELU()
            self.down2 = None

    def forward(self, x):
        # x: (B, in_channels, 64, 64)
        x = self.stem_act(self.stem_norm(self.stem_conv(x)))       # full-res feature extraction
        r = self.res_act(self.res_norm(self.res_conv(x)))
        x = x + r                                                   # residual, still full-res
        x = self.drop(x)

        x = self.down1_act(self.down1_norm(self.down1(x)))
        if self.down2 is not None:
            x = self.down2_act(self.down2_norm(self.down2(x)))
        return x


Writing model/cnn_stem.py


In [9]:
%%writefile model/s4d_recurrent.py
import math
import torch
import torch.nn as nn
from einops import repeat

class S4D(nn.Module):
    """
    Diagonal Structured State Space (S4D) layer.
    
    Implements the S4D variant of Structured State Spaces using diagonal state matrices
    for computational efficiency. This layer processes sequences through a continuous-time
    state space model discretized using the bilinear method, enabling modeling of long-range
    dependencies with linear complexity.
    
    The S4D model parameterizes the state space with:
    - Diagonal complex-valued state transition matrix A
    - Complex-valued output projection matrix C  
    - Skip connection parameter D
    - Learnable discretization timestep dt
    
    Convolution is performed efficiently in the frequency domain using FFT.
    
    Parameters
    ----------
    d_model : int
        Input and output feature dimension (number of independent SSM copies).
    d_state : int, optional
        Latent state dimension (must be even for complex representation). 
        Default is 64.
    dt_min : float, optional
        Minimum discretization timestep. Default is 0.001.
    dt_max : float, optional
        Maximum discretization timestep. Default is 0.1.
    transposed : bool, optional
        If True, expects input shape (B, H, L). If False, expects (B, L, H).
        Default is True.
    lr : float, optional
        Custom learning rate for SSM parameters. If None, uses optimizer default.
        If 0.0, parameters become fixed buffers.
    
    Attributes
    ----------
    h : int
        Number of independent SSM copies (equals d_model).
    n : int
        State dimension.
    log_dt : nn.Parameter or buffer
        Log-space discretization timestep (shape: h).
    log_A_real : nn.Parameter or buffer
        Log-space real part of diagonal state matrix (shape: h, n//2).
    A_imag : nn.Parameter or buffer
        Imaginary part of diagonal state matrix (shape: h, n//2).
    C : nn.Parameter
        Complex output projection matrix (shape: h, n//2, 2 for real view).
    D : nn.Parameter  
        Skip connection weights (shape: h).
    
    Input
    -----
    u : torch.Tensor
        Input sequence of shape (B, H, L) if transposed=True, or (B, L, H) otherwise.
        B : batch size
        H : d_model (feature dimension)
        L : sequence length
    
    Returns
    -------
    y : torch.Tensor
        Output sequence of same shape as input.
    None
        Placeholder for compatibility with stateful interfaces.
    
    References
    ----------
    Gu, A., Goel, K., & Ré, C. (2022). Efficiently Modeling Long Sequences with 
    Structured State Spaces. In ICLR 2022.
    
    Gu, A., Gupta, A., Goel, K., & Ré, C. (2022). On the Parameterization and 
    Initialization of Diagonal State Space Models. In NeurIPS 2022.
    """
    def __init__(self, d_model, d_state=64, dt_min=0.001, dt_max=0.1, transposed=True, lr=None):
        super().__init__()
        self.h = d_model
        self.n = d_state
        self.transposed = transposed

        # --- Initial Parameter Tensors ---
        log_dt = torch.rand(self.h) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
        log_A_real = torch.log(0.5 * torch.ones(self.h, self.n // 2))
        A_imag = math.pi * repeat(torch.arange(self.n // 2), 'n -> h n', h=self.h)
        C_init = torch.randn(self.h, self.n // 2, dtype=torch.cfloat)

        # --- Registration ---
        # We use 'register' to set weight_decay=0.0 and custom LRs for SSM cores
        self.register("log_dt", log_dt, lr)
        self.register("log_A_real", log_A_real, lr)
        self.register("A_imag", A_imag, lr)
        
        # C and D are usually treated as standard parameters
        self.C = nn.Parameter(torch.view_as_real(C_init))
        self.D = nn.Parameter(torch.randn(self.h))

    def register(self, name, tensor, lr=None):
        """
        Register a parameter or buffer with custom optimization settings.
        
        Parameters
        ----------
        name : str
            Name for the parameter/buffer.
        tensor : torch.Tensor
            Tensor to register.
        lr : float, optional
            Custom learning rate. If 0.0, registers as buffer (non-trainable).
            If None, uses optimizer default. Otherwise, attaches custom lr metadata.
        """
        if lr == 0.0:
            self.register_buffer(name, tensor)
        else:
            self.register_parameter(name, nn.Parameter(tensor))
            # Tag the parameter with optimization constraints
            optim = {"weight_decay": 0.0}
            if lr is not None: 
                optim["lr"] = lr
            setattr(getattr(self, name), "_optim", optim)

    def forward(self, u):
        """
        Forward pass through the S4D layer.
        
        Computes the convolution of the input sequence with the SSM kernel using FFT.
        The kernel is generated from the continuous-time SSM parameters and discretized
        using the learned timestep dt.
        
        Process:
        1. Materialize SSM parameters (dt, A, C) from log-space representations
        2. Generate discrete convolution kernel K via truncated power series
        3. Perform FFT-based convolution: y = K * u
        4. Add skip connection: y = y + D * u
        
        Parameters
        ----------
        u : torch.Tensor
            Input sequence of shape (B, H, L) if transposed=True, else (B, L, H).
            B : batch size
            H : feature dimension (d_model)
            L : sequence length
        
        Returns
        -------
        y : torch.Tensor
            Output sequence of same shape as input.
        None
            Placeholder for state (included for interface compatibility).
        """
        if not self.transposed: u = u.transpose(-1, -2)
        L = u.size(-1)

        # 1. Materialize Parameters
        dt = torch.exp(self.log_dt) 
        C = torch.view_as_complex(self.C) 
        A = -torch.exp(self.log_A_real) + 1j * self.A_imag 

        # 2. Generate Kernel K (Diagonal SSM formula)
        dtA = A * dt.unsqueeze(-1)  
        # Power series generation: exp(A * dt * t)
        K_exp = torch.exp(dtA.unsqueeze(-1) * torch.arange(L, device=u.device)) 
        C_tilde = C * (torch.exp(dtA) - 1.) / A
        k = 2 * torch.einsum('hn, hnl -> hl', C_tilde, K_exp).real 

        # 3. FFT Convolution (y = k * u)
        k_f = torch.fft.rfft(k, n=2*L) 
        u_f = torch.fft.rfft(u, n=2*L) 
        y = torch.fft.irfft(u_f * k_f, n=2*L)[..., :L] 

        # 4. Skip Connection
        y = y + u * self.D.unsqueeze(-1)

        if not self.transposed: y = y.transpose(-1, -2)
        return y, None

Writing model/s4d_recurrent.py


In [10]:
%%writefile model/gclassifier.py
import torch
import torch.nn as nn

from .hilbert import HilbertScan
from .tlts import TakeLastTimestep
from .s4d_recurrent import S4D

class GalaxyClassifierS4D(nn.Module):
    """
    Galaxy classifier using Hilbert Scan and S4 sequence modeling.
    
    This model scans 2D galaxy images into a 1D Hilbert sequence, projects
    the multi-channel pixel values to a higher-dimensional feature space,
    processes the sequence with stacked S4 layers with GELU activations, 
    takes the final timestep as a summary representation, and applies a 
    linear classifier to predict galaxy types.
    
    Parameters
    ----------
    s4_state : int, optional
        Hidden state dimension for the S4 layers (default is 64).
    d_model : int, optional
        Output feature dimension of the S4 layers (default is 64).
    num_classes : int, optional
        Number of output classes (default is 4).
    colored : bool, optional
        If True, expects RGB input images (3 channels); if False, expects
        grayscale images (1 channel) (default is True).
    
    Attributes
    ----------
    seq_len : int
        Sequence length after Hilbert scan (64*64 = 4096).
    d_model : int
        Dimension of the S4 output features.
    hilbert_channels : int
        Number of input channels (1 for grayscale, 3 for RGB).
    hilbert_scan : HilbertScan
        Layer that converts 2D images into 1D sequences using a Hilbert scan.
    uproject : nn.Linear
        Linear projection mapping hilbert_channels to d_model dimensions.
    s4_1 : S4D
        First S4 layer.
    act1 : nn.GELU
        GELU activation after the first S4 layer.
    s4_2 : S4D
        Second S4 layer.
    act2 : nn.GELU
        GELU activation after the second S4 layer.
    take_last : TakeLastTimestep
        Layer that extracts the last timestep from the sequence.
    fc : nn.Linear
        Linear classifier mapping S4 features to output classes.
    softmax : nn.Softmax
        Softmax layer for output probabilities.
    """
    def __init__(self, s4_state=64, d_model=64, num_classes=4, colored=True):
        super().__init__()
        self.seq_len = 64 * 64 
        self.d_model = d_model

        # Hilbert Scan layer
        self.hilbert_scan = HilbertScan()
        self.hilbert_channels = 1 if not colored else 3

        self.uproject = nn.Linear(self.hilbert_channels, d_model)

        # S4 layers -- recurrent, not the old FFT/causal-conv layer. Verified
        # against the trained portable first (see recurrent_vs_causal_conv_verification.png):
        # logits matched to ~6e-4, same argmax on every sample. Conv layer's gone now.
        self.s4_1 = S4D(d_model=d_model, d_state=s4_state, transposed=False)
        self.act1 = nn.GELU()

        self.s4_2 = S4D(d_model=d_model, d_state=s4_state, transposed=False)
        self.act2 = nn.GELU()

        # Take last timestep
        self.take_last = TakeLastTimestep()

        # Classifier
        self.fc = nn.Linear(d_model, num_classes)

        # Softmax for output probabilities
        self.softmax = nn.Softmax(dim=-1)


       # -------------------------------------------------------------------------
        # PARAMETER COUNT VERIFICATION (Task 8.4)
        # Verified with torchinfo.summary()
        # -------------------------------------------------------------------------
        # 1. Input Projection: (1 * 64) + 64 = 128 params
        # 2. S4D Layer 1 (Optimized N/2 symmetry): 
        #    Per feature: 130 params (vs 258 naive)
        #    Total: 64 * 130 = 8,320 params
        # 3. S4D Layer 2: Same as Layer 1 = 8,320 params
        # 4. Classifier Head: (64 * 4) + 4 = 260 params
        # 
        # GRAND TOTAL: 128 + 8,320 + 8,320 + 260 = 17,028 Parameters
        # -------------------------------------------------------------------------



        # -------------------------------------------------------------------------
        # FLOPS ESTIMATION (Task 8.5) -- redone for the recurrent S4D layer
        # Sequence Length L = 4096, d_model = 64, d_state = 64, C = 1
        # -------------------------------------------------------------------------
        # 1. Input Projection: L * C * d_model
        #    4096 * 1 * 64 = 262,144 Ops
        #
        # 2. S4D Layers (x2): no more FFT kernel, so no log(L) term. Each layer
        #    steps through L timesteps, and at each step does ~2 complex MACs per
        #    state element (one for the state update, one for the output sum) --
        #    a complex MAC costs roughly 4x a real one, call it ~8 real ops:
        #    2 * (L * (d_state/2) * d_model * 8) = 2 * (4096*32*64*8) ≈ 134.2M Ops
        #
        # 3. Classifier Head: d_model * Classes
        #    64 * 4 = 256 Ops
        #
        # GRAND TOTAL: ~134.5 Million Operations per forward pass
        # (vs. ~6.55M under the old FFT estimate -- more raw arithmetic, since
        # we lost the O(log L) speedup, but no transcendental-heavy kernel
        # generation either, which is most of why it still benchmarks faster
        # in practice at this d_model -- see model/s4d_recurrent.py)
        # -------------------------------------------------------------------------

    def forward(self, x, return_logits=False):
        """
        Forward pass of the PixelS4Galaxy model.
        
        Parameters
        ----------
        x : torch.Tensor
            Input tensor of shape (B, C, 64, 64), where B is the batch size
            and C is the number of channels (1 for grayscale, 3 for RGB).
        return_logits : bool, optional
            If True, returns raw logits instead of softmax probabilities 
            (default is False).
        
        Returns
        -------
        output : torch.Tensor
            If return_logits=True: Output logits of shape (B, num_classes),
            representing unnormalized scores for each galaxy class.
            If return_logits=False: Output probabilities of shape (B, num_classes),
            representing the softmax probability distribution over classes.
        """
        B, C, H, W = x.shape
        assert H == 64 and W == 64, "Expected 64x64"
        assert C == self.hilbert_channels, f"Expected {self.hilbert_channels} channels"

        # 1. Hilbert scan: 2D > 1D
        x_seq = self.hilbert_scan(x)  # (B,4096,C)

        # 2. Input projection: C > d_model
        x_proj = self.uproject(x_seq)  # (B,4096,d_model)

        # 3. S4D layer 1 + GELU
        s4_out1, _ = self.s4_1(x_proj)
        a1 = self.act1(s4_out1)  # (B,4096,d_model

        # 4. S4D layer 2 + GELU
        s4_out2, _ = self.s4_2(a1)
        a2 = self.act2(s4_out2)      # (B,4096,d_model)

        # 5. Take last timestep
        last = self.take_last(a2)          # (B,d_model)

        # 6. Classifier: d_model > num_classes
        logits = self.fc(last)             # (B,4)

        # Return logits or softmax
        if return_logits:
            return logits
        return self.softmax(logits)

# basically this function takes the image and turns it into a sequence
        # first we check the shape to make sure its 64x64
        # then the hilbert scan flattens the 2D image into a long 1D list of pixels
        # after that we project it up to hidden size using a linear layer
        # then it goes through two S4 layers with GELU activation in between to learn features
        # since its a sequence model we only care about the very last timestep which has the summary
        # finally we pass that last step to the linear classifier to get the 4 class scores
        # and if we need probs we apply softmax otherwise just return the raw logits

        #raise NotImplementedError("Forward method not implemented yet.")

Writing model/gclassifier.py


In [11]:
%%writefile model/gclassifier_hybrid.py
import torch
import torch.nn as nn

from .cnn_stem import CNNStem
from .hilbert import HilbertScan
from .tlts import TakeLastTimestep
from .s4d_recurrent import S4D


class GalaxyClassifierCNNS4D(nn.Module):
    """
    CNN-stem -> S4D hybrid galaxy classifier.

    Companion/competitor to GalaxyClassifierS4D (model/gclassifier.py). The
    baseline assumes long-range pixel dependency matters for galaxy
    morphology (it scans the full 4096-pixel image straight into S4D). This
    model tests the opposite hypothesis: that morphology is dominated by
    local structure (arm curvature, edge sharpness, blob shape), so a small
    CNN stem can do local feature extraction + spatial downsampling first,
    handing S4D a much shorter sequence, while preserving accuracy.

    image (B,C,64,64)
      -> CNNStem                          -> (B, d_model, grid, grid)
      -> HilbertScan(n=grid)              -> (B, grid*grid, d_model)
      -> S4D(d_model, d_state) s4_1       -> (B, grid*grid, d_model)
      -> GELU
      -> S4D(d_model, d_state) s4_2       -> (B, grid*grid, d_model)
      -> GELU
      -> TakeLastTimestep                 -> (B, d_model)
      -> Linear(d_model, num_classes) fc  -> (B, num_classes)
      -> softmax (or raw logits, matching GalaxyClassifierS4D's API exactly)

    With the default stem_reduction=16, grid=16, so seq_len=256 -- a 16x cut
    from the baseline's 4096, and therefore ~16x fewer S4D-loop ops (S4D's
    per-layer cost is O(d_model * seq_len * d_state/2), linear in seq_len --
    see the FLOPS comment in model/gclassifier.py for the exact op-count
    formula this scales).

    The CNN stem's last conv already projects channels up to d_model, so --
    unlike GalaxyClassifierS4D -- there is no separate `uproject` Linear
    here; the stem's output channel dim *is* the projection.

    S4D itself (model/s4d_recurrent.py) is reused unmodified apart from an
    optional 3rd stacked layer (num_s4_layers=3); d_state is still
    configurable via s4_state, only seq_len shrinks because of what
    happens upstream in the stem.

    Parameters
    ----------
    s4_state : int, optional
        Hidden state dimension for the S4D layers (default 64).
    d_model : int, optional
        Output feature dimension of the CNN stem / S4D layers (default 64).
    num_classes : int, optional
        Number of output classes (default 4).
    colored : bool, optional
        If True, expects RGB input images (3 channels); if False, expects
        grayscale images (1 channel). Default True -- color carries the
        dust-lane/reddening signal needed to separate Smooth Cigar from
        Edge-on Disk, the dominant error mode observed with grayscale-only
        input.
    stem_reduction : int, optional
        Sequence-length reduction factor applied by the CNN stem before
        Hilbert-scanning, one of {4, 16}:
          - 16 (default): three-stage stem (stride1 -> stride2 -> stride2),
            64x64 -> 16x16, seq_len 4096 -> 256.
          - 4: milder cut, two-stage stem (stride1 -> stride2),
            64x64 -> 32x32, seq_len 4096 -> 1024. Retains more spatial
            resolution; the recommended default for research runs where
            accuracy matters more than compute savings.
    mid_channels : int, optional
        Hidden channel width inside the stem's stride-1 detail-extraction
        stage. Default 32 (raised from the course version's 16 for more
        capacity).
    stem_dropout : float, optional
        Dropout2d applied inside the stem after the stride-1 block.
        Default 0.1.
    head_dropout : float, optional
        Dropout applied to the pooled sequence representation right before
        the classifier head. Default 0.2.
    num_s4_layers : int, optional
        Number of stacked S4D layers, one of {2, 3}. Default 2 (matches
        the course architecture); set to 3 to test whether 2 layers is a
        capacity bottleneck once the stem/data changes above are in place.

    Attributes
    ----------
    seq_len : int
        Sequence length after the CNN stem + Hilbert scan (256 for
        stem_reduction=16, 1024 for stem_reduction=4).
    d_model : int
        Dimension of the S4D output features.
    hilbert_channels : int
        Number of input image channels (1 for grayscale, 3 for RGB).
    cnn_stem : CNNStem
        Conv stem doing local feature extraction, downsampling, and
        channel projection to d_model.
    hilbert_scan : HilbertScan
        Scans the stem's (B, d_model, grid, grid) feature map into a 1D
        sequence via a Hilbert curve over the grid.
    s4_1, s4_2 : S4D
        Stacked S4D layers (unmodified from the baseline).
    act1, act2 : nn.GELU
    take_last : TakeLastTimestep
    fc : nn.Linear
    softmax : nn.Softmax
    """

    def __init__(self, s4_state=64, d_model=64, num_classes=4, colored=True,
                 stem_reduction=16, mid_channels=32, stem_dropout=0.1,
                 head_dropout=0.2, num_s4_layers=2):
        super().__init__()
        if stem_reduction not in (4, 16):
            raise ValueError(f"stem_reduction must be 4 or 16, got {stem_reduction}")
        if num_s4_layers not in (2, 3):
            raise ValueError(f"num_s4_layers must be 2 or 3, got {num_s4_layers}")

        self.hilbert_channels = 1 if not colored else 3
        self.d_model = d_model
        self.stem_reduction = stem_reduction
        self.num_s4_layers = num_s4_layers

        # Spatial side of the feature grid after the stem: 64 -> 64/sqrt(reduction)
        # reduction=16 -> two stride-2 blocks -> /4 side reduction -> grid=16
        # reduction=4  -> one stride-2 block   -> /2 side reduction -> grid=32
        grid = 64 // (4 if stem_reduction == 16 else 2)
        self.seq_len = grid * grid

        # CNN stem: local feature extraction + downsampling. Its last conv
        # projects channels to d_model, so no separate uproject Linear is
        # needed (unlike the baseline). Research-version stem (see
        # cnn_stem.py) runs a full-resolution stride-1 pass before any
        # downsampling, so thin structures (dust lanes etc.) survive.
        self.cnn_stem = CNNStem(
            in_channels=self.hilbert_channels,
            d_model=d_model,
            mid_channels=mid_channels,
            reduction=stem_reduction,
            dropout=stem_dropout,
        )

        # Hilbert scan over the downsampled feature grid (not the raw image)
        self.hilbert_scan = HilbertScan(n=grid)

        # S4D layers. Stack of 2 (course default) or 3 (research option --
        # a bit more sequence-modeling capacity now that d_model/mid_channels
        # are also bigger, in case 2 layers is now the bottleneck).
        self.s4_1 = S4D(d_model=d_model, d_state=s4_state, transposed=False)
        self.act1 = nn.GELU()

        self.s4_2 = S4D(d_model=d_model, d_state=s4_state, transposed=False)
        self.act2 = nn.GELU()

        if num_s4_layers == 3:
            self.s4_3 = S4D(d_model=d_model, d_state=s4_state, transposed=False)
            self.act3 = nn.GELU()
        else:
            self.s4_3 = None
            self.act3 = None

        # Take last timestep (currently mean-pooling, see tlts.py)
        self.take_last = TakeLastTimestep()

        # Light dropout before the classifier head -- regularization for
        # the ~8k-image dataset now that capacity has gone up.
        self.head_drop = nn.Dropout(head_dropout)

        # Classifier
        self.fc = nn.Linear(d_model, num_classes)

        # Softmax for output probabilities
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, return_logits=False):
        """
        Forward pass of the CNN-stem -> S4D hybrid model.

        Parameters
        ----------
        x : torch.Tensor
            Input tensor of shape (B, C, 64, 64), where B is the batch size
            and C is the number of channels (1 for grayscale, 3 for RGB).
        return_logits : bool, optional
            If True, returns raw logits instead of softmax probabilities
            (default is False).

        Returns
        -------
        output : torch.Tensor
            If return_logits=True: Output logits of shape (B, num_classes).
            If return_logits=False: Output probabilities of shape
            (B, num_classes), the softmax distribution over classes.
        """
        B, C, H, W = x.shape
        assert H == 64 and W == 64, "Expected 64x64"
        assert C == self.hilbert_channels, f"Expected {self.hilbert_channels} channels"

        # 1. CNN stem: local feature extraction + spatial downsampling,
        #    also projects channels -> d_model
        feat = self.cnn_stem(x)  # (B, d_model, grid, grid)

        # 2. Hilbert scan: 2D feature map -> 1D sequence
        x_seq = self.hilbert_scan(feat)  # (B, seq_len, d_model)

        # 3. S4D layer 1 + GELU
        s4_out1, _ = self.s4_1(x_seq)
        a1 = self.act1(s4_out1)  # (B, seq_len, d_model)

        # 4. S4D layer 2 + GELU
        s4_out2, _ = self.s4_2(a1)
        a2 = self.act2(s4_out2)  # (B, seq_len, d_model)

        # 4b. Optional S4D layer 3 + GELU (num_s4_layers=3)
        if self.s4_3 is not None:
            s4_out3, _ = self.s4_3(a2)
            a2 = self.act3(s4_out3)  # (B, seq_len, d_model)

        # 5. Take last timestep (mean-pool, see tlts.py)
        last = self.take_last(a2)  # (B, d_model)
        last = self.head_drop(last)

        # 6. Classifier: d_model -> num_classes
        logits = self.fc(last)  # (B, num_classes)

        # Return logits or softmax
        if return_logits:
            return logits
        return self.softmax(logits)


Writing model/gclassifier_hybrid.py


In [12]:
%%writefile model/gclassifier_cnn_only.py
import torch
import torch.nn as nn

from .cnn_stem import CNNStem


class GalaxyClassifierCNNOnly(nn.Module):
    """
    CNN-only baseline: same CNNStem as GalaxyClassifierCNNS4D, but with NO
    S4D layers at all. Feature map is pooled directly (global average
    pooling) and classified.

    Why this model exists
    ----------------------
    Every result so far has tested "CNN stem -> S4D". Nothing so far has
    tested "CNN alone" at a matched parameter budget. That leaves an open
    question raised directly by the TA:

      1. Would a ~43K-param CNN alone reach similar accuracy to the
         ~55K-param CNN+S4D hybrid (86.80%)?
      2. Would a ~60K-param CNN alone reach similar accuracy to the
         ~63K-param CNN+S4D 3-layer hybrid (86.65%)?
      3. Is S4D actually adding signal beyond what the CNN stem alone
         already provides, or is the CNN stem doing all the work (echoing
         the color ablation finding, where the "obvious" explanation
         wasn't the real one)?
      4. How small can the CNN get before accuracy collapses -- i.e.
         where is the actual capacity floor for this task?

    This class, together with scripts/train_cnn_only.py, runs three sizes
    (~10K, ~43K, ~60K params) to answer all four questions with real
    numbers instead of assumptions on either side.

    Design choice: global average pooling over the stem's output grid is
    used as the CNN-only readout, because it is the direct analog of the
    S4D hybrid's mean-pooling over the Hilbert-scanned sequence -- same
    "average all spatial/sequence positions" idea, just without S4D's
    sequential processing in between. This keeps the comparison about
    S4D specifically, not about the readout strategy also changing.

    Parameters
    ----------
    num_classes : int, optional
        Number of output classes (default 4).
    colored : bool, optional
        RGB (3-channel) if True, grayscale (1-channel) if False. Default True.
    stem_reduction : int, optional
        Passed through to CNNStem, one of {4, 16}. Default 16 (matches the
        winning hybrid configuration).
    mid_channels : int, optional
        Stem hidden channel width. Default 32.
    d_model : int, optional
        Stem output channel width (and refine-conv width, if used). Default 64.
    stem_dropout : float, optional
        Dropout2d inside the stem. Default 0.1.
    head_dropout : float, optional
        Dropout applied to the pooled vector before the classifier head. Default 0.2.
    use_refine_conv : bool, optional
        If True, adds one extra 1x1 conv + GroupNorm + GELU after the stem,
        before pooling -- used to hit the ~43K/~60K parameter targets
        without changing the stem's own architecture. Default True.
    """

    def __init__(self, num_classes=4, colored=True, stem_reduction=16,
                 mid_channels=32, d_model=64, stem_dropout=0.1,
                 head_dropout=0.2, use_refine_conv=True):
        super().__init__()
        if stem_reduction not in (4, 16):
            raise ValueError(f"stem_reduction must be 4 or 16, got {stem_reduction}")

        self.hilbert_channels = 1 if not colored else 3
        self.d_model = d_model
        self.stem_reduction = stem_reduction
        self.use_refine_conv = use_refine_conv

        grid = 64 // (4 if stem_reduction == 16 else 2)
        self.seq_len = grid * grid  # kept for API/logging parity with the hybrid model

        self.cnn_stem = CNNStem(
            in_channels=self.hilbert_channels,
            d_model=d_model,
            mid_channels=mid_channels,
            reduction=stem_reduction,
            dropout=stem_dropout,
        )

        if use_refine_conv:
            # 1x1 conv: adds a small amount of extra depth/capacity so the
            # small/large variants can be tuned to specific parameter
            # budgets (~43K, ~60K) without changing the stem's own
            # architecture (which is the part already validated by the
            # earlier ablation).
            self.refine_conv = nn.Conv2d(d_model, d_model, kernel_size=1)
            groups = 8 if d_model % 8 == 0 else 1
            self.refine_norm = nn.GroupNorm(groups, d_model)
            self.refine_act = nn.GELU()
        else:
            self.refine_conv = None

        self.global_pool = nn.AdaptiveAvgPool2d(1)  # (B, d_model, H, W) -> (B, d_model, 1, 1)
        self.head_drop = nn.Dropout(head_dropout)
        self.fc = nn.Linear(d_model, num_classes)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, return_logits=False):
        # x: (B, hilbert_channels, 64, 64)
        feat = self.cnn_stem(x)  # (B, d_model, grid, grid)

        if self.refine_conv is not None:
            feat = self.refine_act(self.refine_norm(self.refine_conv(feat)))

        pooled = self.global_pool(feat).flatten(1)  # (B, d_model)
        pooled = self.head_drop(pooled)

        logits = self.fc(pooled)  # (B, num_classes)

        if return_logits:
            return logits
        return self.softmax(logits)


Writing model/gclassifier_cnn_only.py


In [13]:
%%writefile model/functions.py
import os
import numpy as np

# PyTorch
import torch.nn.functional as F

# GalaxyMNIST dataset
from galaxy_mnist import GalaxyMNIST 

def load_data(root: str, download: bool = True, train: bool = True, colored: bool = False):
    """Load and preprocess GalaxyMNIST dataset.
    
    Parameters:
    -----------
    root : str
        Root directory where the dataset is stored or will be downloaded.
    download : bool
        Whether to download the dataset if not present.
    train : bool
        Whether to load the training set (True) or test set (False).
    colored : bool, optional
        Whether to use colored images (3 channels) or grayscale (1 channel).
        (default is False)
           
    Returns:
    --------
    X : torch.Tensor
        Preprocessed images of shape (N, 1, 64, 64) with pixel values in [0, 1] if grayscale,
        or (N, 3, 64, 64) if colored.
    y_onehot : torch.Tensor
        One-hot encoded labels of shape (N, num_classes).
    y : torch.Tensor
        Original labels of shape (N,).
    """
    dataset = GalaxyMNIST(root=root, download=download, train=train)
    print(f"Original Dataset Size: {len(dataset.data)} samples")

    # 1. Extract and process images: Mean across channels, Normalize to [0, 1]
    # Data shape is (N, 3, 64, 64) -> (N, 1, 64, 64)
    X = dataset.data.float()           # convert from uint8 -> float
    if not colored:
        X = X.mean(dim=1, keepdim=True)  # convert to grayscale by averaging channels
    X = X / 255.0                     # normalize to [0, 1]

    # 2. Extract targets and convert to one-hot encoding
    y = dataset.targets.long()
    # One hot encode the labels
    y_onehot = F.one_hot(y).float()
    return X, y_onehot, y

def format_row(values):
    """Formats a list of values, comma separated, even width"""
    return ", ".join(f"{v:10.6f}" if isinstance(v, (float, np.float32, np.float64)) else f"{v}" for v in values)

def export_model_parameters(model, output_dir="galaxy_s4_model_params"):
    """
    General function to export all model parameters AND buffers to .bin and .txt.
    Logic: (A, B, C) -> A blocks of B rows x C cols.
    Follows natural Row-Major storage order.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    txt_path = os.path.join(output_dir, "weights.txt")
    bin_path = os.path.join(output_dir, "weights.bin")

    print(f"--- Exporting Model: {model.__class__.__name__} ---")

    state_dict = model.state_dict()

    with open(txt_path, "w") as f_txt, open(bin_path, "wb") as f_bin:
        for name, tensor in state_dict.items():
            shape = list(tensor.shape)
            print(f"Saving: {name:40} | Shape: {shape}")
            
            data_np = tensor.detach().cpu().contiguous().numpy()
            f_bin.write(data_np.astype(np.float32).tobytes())

            f_txt.write(f"[{name}] Shape: {shape}\n")

            if len(shape) == 0:
                f_txt.write(f"{data_np.item():10.6f}\n\n")

            elif len(shape) == 1:
                f_txt.write(format_row(data_np) + "\n\n")

            elif len(shape) == 2:
                rows, cols = shape
                for r in range(rows):
                    f_txt.write(format_row(data_np[r]) + "\n")
                f_txt.write("\n")

            elif len(shape) == 3:
                A, B, C = shape
                for a in range(A):
                    f_txt.write(f"# Block {a}\n")
                    for b in range(B):
                        row_values = data_np[a, b, :]
                        f_txt.write(format_row(row_values) + "\n")
                    f_txt.write("\n")
            
            elif len(shape) == 4:
                A, B, C, D = shape
                for a in range(A):
                    for b in range(B):
                        f_txt.write(f"# Block {a}, {b}\n")
                        for c in range(C):
                            f_txt.write(format_row(data_np[a, b, c, :]) + "\n")
                        f_txt.write("\n")

            else:
                f_txt.write(format_row(data_np.flatten()) + "\n\n")

    print(f"--- Export Complete. Files located in '{output_dir}' ---")

Writing model/functions.py


In [14]:
%%writefile model/interface.py
import torch

class ModelInterface:
    """
    Unified interface for galaxy classification models.
    
    This class abstracts the implementation details (Python vs RISC-V) and provides
    a consistent API for model inference regardless of backend.
    
    Parameters
    ----------
    implementation : str
        Either 'python' or 'riscv'.
    model_path : str
        Path to model weights (used for Python implementation).
    num_classes : int
        Number of output classes.
    colored : bool
        Whether model expects colored or grayscale images.
    device : torch.device
        Device for inference.
    
    Methods
    -------
    __call__(x)
        Run inference on input tensor x.
    """
    
    def __init__(self, implementation, model_path, num_classes, colored, device):
        """Initialize model based on implementation type."""
        self.implementation = implementation
        self.device = device
        
        if implementation == 'python':
            from model import GalaxyClassifierS4D
            print(f"Loading Python model from {model_path}")
            self.model = GalaxyClassifierS4D(colored=colored).to(device)
            self.model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
            self.model.eval()
            
        elif implementation == 'riscv':
            print("Initializing RISC-V interface")
            # TODO: Setup RISC-V communication/configuration
            self.model = None
    
    def __call__(self, x):
        """
        Run model inference.
        
        Parameters
        ----------
        x : torch.Tensor
            Input tensor.
        
        Returns
        -------
        torch.Tensor
            Model predictions.
        """
        if self.implementation == 'python':
            return self.model(x)
        elif self.implementation == 'riscv':
            # TODO: Send x to RISC-V QEMU, receive predictions
            raise NotImplementedError("RISC-V inference not yet implemented")
    
    def eval(self):
        """Set model to evaluation mode (for consistency with PyTorch API)."""
        if self.implementation == 'python':
            self.model.eval()


Writing model/interface.py


In [15]:
%%writefile model/gui.py
"""
Interactive Galaxy Explorer GUI

This interactive visualization tool allows you to browse through the validation set and examine the 
model's predictions in real-time. The GUI displays each galaxy image using the Magma colormap 
(commonly used in astronomy visualization) alongside the model's softmax probability distribution
across all four classes.

Controls
--------
- LEFT/RIGHT Arrow Keys: Navigate through validation samples
- R Key: Jump to a random sample
- M Key: Toggle Magma colormap on/off
- Q Key: Quit the application

The visualization highlights the predicted class with a green bar, making it easy to spot correct 
classifications and identify failure cases where the model might confuse similar morphologies 
(e.g., smooth round vs. smooth cigar galaxies).
"""

import os
os.environ['PYGAME_HIDE_SUPPORT_PROMPT'] = '1'

import pygame
import numpy as np
import torch
import matplotlib.pyplot as plt

class GalaxyExplorerGUI:
    """
    Interactive GUI for exploring galaxy classifications.
    
    Displays galaxy images alongside model predictions with real-time navigation.
    Supports toggling between standard RGB and Magma colormap visualization.
    
    Parameters
    ----------
    model : ModelInterface
        Trained model for galaxy classification.
    x_val : torch.Tensor
        Validation images tensor of shape (N, C, H, W).
    y_val : torch.Tensor
        One-hot encoded validation labels of shape (N, num_classes).
    device : torch.device
        Device for running inference (CPU or CUDA).
    
    Attributes
    ----------
    current_idx : int
        Index of currently displayed sample.
    num_samples : int
        Total number of validation samples.
    predictions : np.ndarray
        Current model prediction probabilities.
    use_magma : bool
        Whether to apply Magma colormap to displayed image.
    """
    def __init__(self, model, x_val, y_val, device):
        self.model = model
        self.x_val = x_val  
        self.y_val = y_val  
        self.device = device
        
        self.current_idx = 0
        self.num_samples = len(x_val)
        self.predictions = np.zeros(4)
        
        # New toggle state
        self.use_magma = False
        
        pygame.init()
        self.WIDTH, self.HEIGHT = 1000, 600
        self.screen = pygame.display.set_mode((self.WIDTH, self.HEIGHT))
        pygame.display.set_caption("S4 GALAXY EXPLORER")
        
        self.CANVAS_SIZE = 448
        self.class_labels =  ["Smooth Round", "Smooth Cigar", "Edge-on Disk", "Unbarred Spiral"] # Class names for GalaxyMNIST
        
        self.COLOR_BG = (5, 5, 8)
        self.COLOR_ACCENT = (255, 160, 60) 
        self.COLOR_SUCCESS = (0, 255, 120)
        self.COLOR_FAILURE = (255, 50, 50)
        
        self.font = pygame.font.SysFont("monospace", 15)
        self.big_font = pygame.font.SysFont("monospace", 22, bold=True)
        
        self.cmap = plt.get_cmap('magma')
        self.update_sample(0)

    def update_sample(self, delta):
        """
        Update the currently displayed sample and compute predictions.
        
        Parameters
        ----------
        delta : int
            Offset to add to current index (wraps around at boundaries).
        """
        self.current_idx = (self.current_idx + delta) % self.num_samples
        self.model.eval()
        with torch.no_grad():
            img_tensor = self.x_val[self.current_idx].unsqueeze(0).to(self.device)
            probs = self.model(img_tensor)
            self.predictions = probs.squeeze().cpu().numpy()

    def draw(self):
        """
        Render the current frame of the GUI.
        
        Displays the galaxy image (with optional Magma colormap), prediction bars,
        sample metadata, and keyboard controls. Highlights correct predictions in
        green and incorrect predictions in red.
        """
        self.screen.fill(self.COLOR_BG)
        
        # x_val is always (N, C, H, W) -- so raw_img here is always channel-first,
        # never (H, W, C). The "or [H, W, C]" in the old comment was never actually
        # reachable, and assuming it was is what broke grayscale rendering below.
        raw_img = self.x_val[self.current_idx].numpy()
        
        if self.use_magma:
            # mean over the channel axis works whether C=1 (trivial mean, same as
            # squeezing) or C=3 (proper RGB->gray average) -- no need to special-case
            gray_img = raw_img.mean(axis=0)
            
            magma_img = self.cmap(gray_img) 
            rgb_render = (magma_img[:, :, :3] * 255).astype(np.uint8)
        else:
            # Standard RGB Render
            # Pygame wants (H, W, 3) uint8. Grayscale data is (1, H, W) -- transposing
            # alone gives (H, W, 1), which pygame's make_surface rejects (last dim must
            # be exactly 3), so we tile the single channel across R/G/B instead.
            rgb_render = raw_img.transpose(1, 2, 0)  # (C, H, W) -> (H, W, C)
            if rgb_render.shape[-1] == 1:
                rgb_render = np.repeat(rgb_render, 3, axis=-1)
            
            # Ensure uint8 [0, 255]
            if rgb_render.max() <= 1.0:
                rgb_render = (rgb_render * 255).astype(np.uint8)

        # 2. Create Pygame surface (Transpose from Row-Major to Width-Major)
        surface = pygame.surfarray.make_surface(rgb_render.transpose(1, 0, 2))
        scaled_img = pygame.transform.scale(surface, (self.CANVAS_SIZE, self.CANVAS_SIZE))
        self.screen.blit(scaled_img, (40, 60))

        pygame.draw.rect(self.screen, self.COLOR_ACCENT, (40, 60, self.CANVAS_SIZE, self.CANVAS_SIZE), 2)
        
        true_label_idx = torch.argmax(self.y_val[self.current_idx]).item()
        meta_txt = self.font.render(f"Sample: {self.current_idx} | Truth: {self.class_labels[true_label_idx]} | Magma: {'ON' if self.use_magma else 'OFF'}", True, (200, 200, 200))
        self.screen.blit(meta_txt, (40, 35))

        x_off = 520
        top_pred = np.argmax(self.predictions)
        self.screen.blit(self.big_font.render("MODEL PREDICTION", True, self.COLOR_ACCENT), (x_off, 60))
        
        for i, label in enumerate(self.class_labels):
            prob = self.predictions[i]
            bar_y = 120 + (i * 60)

            if i == top_pred and i == true_label_idx:
                color = self.COLOR_SUCCESS
            elif i == top_pred and i != true_label_idx:
                color = self.COLOR_FAILURE
            else:
                color = (150, 150, 150)

            txt = self.font.render(f"{label}: {prob*100:4.1f}%", True, color)
            self.screen.blit(txt, (x_off, bar_y))
            
            pygame.draw.rect(self.screen, (20, 20, 30), (x_off, bar_y + 25, 400, 15))
            pygame.draw.rect(self.screen, color, (x_off, bar_y + 25, int(prob * 400), 15))

        footer = self.font.render("[L/R] Change | [R] Rand | [M] Magma | [Q] Quit", True, self.COLOR_ACCENT)
        self.screen.blit(footer, (40, 530))

    def run(self):
        """
        Main event loop for the GUI application.
        
        Handles keyboard input for navigation (arrow keys, R for random),
        colormap toggling (M key), and quitting (Q key). Runs until the
        user closes the window or presses Q.
        """
        running = True
        while running:
            for event in pygame.event.get():
                if event.type == pygame.QUIT: running = False
                if event.type == pygame.KEYDOWN:
                    if event.key == pygame.K_RIGHT: self.update_sample(1)
                    if event.key == pygame.K_LEFT: self.update_sample(-1)
                    if event.key == pygame.K_r: self.update_sample(np.random.randint(0, self.num_samples))
                    if event.key == pygame.K_m: self.use_magma = not self.use_magma # Toggle
                    if event.key == pygame.K_q: running = False
            self.draw()
            pygame.display.flip()
        pygame.quit()



Writing model/gui.py


In [16]:
import inspect
from model.s4d_recurrent import S4D

src = inspect.getsource(S4D.forward)
print(src)
assert "torch.fft.rfft" in src and "torch.fft.irfft" in src, "Expected FFT convolution in S4D.forward"
print("CONFIRMED: S4D.forward (used by all four scripts) convolves via torch.fft.rfft/irfft —")
print("the same algorithm as S4DConv in notebook-best-s4d-model_1_.ipynb. No change was needed here.")


    def forward(self, u):
        """
        Forward pass through the S4D layer.
        
        Computes the convolution of the input sequence with the SSM kernel using FFT.
        The kernel is generated from the continuous-time SSM parameters and discretized
        using the learned timestep dt.
        
        Process:
        1. Materialize SSM parameters (dt, A, C) from log-space representations
        2. Generate discrete convolution kernel K via truncated power series
        3. Perform FFT-based convolution: y = K * u
        4. Add skip connection: y = y + D * u
        
        Parameters
        ----------
        u : torch.Tensor
            Input sequence of shape (B, H, L) if transposed=True, else (B, L, H).
            B : batch size
            H : feature dimension (d_model)
            L : sequence length
        
        Returns
        -------
        y : torch.Tensor
            Output sequence of same shape as input.
        None
            Placeholder for

/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [17]:
import torch
from model.tlts import TakeLastTimestep

layer = TakeLastTimestep()
x = torch.randn(3, 6, 2)
out = layer(x)
matches_last = torch.allclose(x[:, -1, :], out)
matches_mean = torch.allclose(x.mean(dim=1), out)
print(f"Output matches x[:, -1, :] (last timestep): {matches_last}")
print(f"Output matches x.mean(dim=1)  (mean pooling): {matches_mean}")
assert matches_last and not matches_mean, "TakeLastTimestep should take the last timestep, not the mean"
print("\nCONFIRMED: TakeLastTimestep now does genuine last-timestep pooling, matching its name/docstring")
print("and the reference notebook — not the mean-pooling it silently did in the uploaded zip.")


Output matches x[:, -1, :] (last timestep): True
Output matches x.mean(dim=1)  (mean pooling): False

CONFIRMED: TakeLastTimestep now does genuine last-timestep pooling, matching its name/docstring
and the reference notebook — not the mean-pooling it silently did in the uploaded zip.


In [18]:
# Sanity check: every model class these scripts use instantiates and forward-passes cleanly.
import torch
from model import GalaxyClassifierS4D, GalaxyClassifierCNNS4D, GalaxyClassifierCNNOnly

def count_params(m):
    return sum(p.numel() for p in m.parameters())

x_rgb = torch.randn(2, 3, 64, 64)
for name, m in [
    ("GalaxyClassifierS4D",     GalaxyClassifierS4D(colored=True)),
    ("GalaxyClassifierCNNS4D",  GalaxyClassifierCNNS4D(colored=True)),
    ("GalaxyClassifierCNNOnly", GalaxyClassifierCNNOnly(colored=True)),
]:
    out = m(x_rgb, return_logits=True)
    print(f"{name:24s} {count_params(m):>8,} params   out shape {tuple(out.shape)}")


GalaxyClassifierS4D        17,156 params   out shape (2, 4)
GalaxyClassifierCNNS4D     55,108 params   out shape (2, 4)
GalaxyClassifierCNNOnly    42,756 params   out shape (2, 4)


In [19]:
import math
import torch
import torch.nn as nn
from einops import repeat

class HilbertScan(nn.Module):
    """Reorders patches of a (B, C, H, W) image along a Hilbert curve."""
    def __init__(self, image_size=64, patch_size=1):
        super().__init__()
        assert image_size % patch_size == 0, "image_size must be divisible by patch_size"
        self.image_size = image_size
        self.patch_size = patch_size
        self.grid_size = image_size // patch_size
        self.num_patches = self.grid_size ** 2
        self.register_buffer("indices", self._get_hilbert_indices(self.grid_size))

    @staticmethod
    def _rot(s, x, y, rx, ry):
        if ry == 0:
            if rx == 1:
                x = s - 1 - x
                y = s - 1 - y
            x, y = y, x
        return x, y

    def _d2xy(self, n, d):
        x = y = 0
        t, s = d, 1
        while s < n:
            rx = (t // 2) & 1
            ry = (t ^ rx) & 1
            x, y = self._rot(s, x, y, rx, ry)
            x += s * rx
            y += s * ry
            t //= 4
            s *= 2
        return x, y

    def _get_hilbert_indices(self, grid_size):
        indices = []
        for d in range(grid_size * grid_size):
            x, y = self._d2xy(grid_size, d)
            indices.append(y * grid_size + x)
        return torch.LongTensor(indices)

    def forward(self, x):
        B, C, H, W = x.shape
        p = self.patch_size
        patches = x.unfold(2, p, p).unfold(3, p, p)
        patches = patches.permute(0, 2, 3, 1, 4, 5).contiguous()
        patches = patches.view(B, self.num_patches, C * p * p)
        return patches[:, self.indices, :]


class TakeLastTimestep(nn.Module):
    def forward(self, x):
        return x[:, -1, :]


class S4DConv(nn.Module):
    """Fast FFT-based parallel convolution S4D layer."""
    def __init__(self, d_model, d_state=64, dt_min=0.001, dt_max=0.1, transposed=True, lr=None):
        super().__init__()
        self.h = d_model
        self.n = d_state
        self.transposed = transposed

        log_dt = torch.rand(self.h) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
        log_A_real = torch.log(0.5 * torch.ones(self.h, self.n // 2))
        A_imag = math.pi * repeat(torch.arange(self.n // 2), 'n -> h n', h=self.h)
        C_init = torch.randn(self.h, self.n // 2, dtype=torch.cfloat)

        self.register("log_dt", log_dt, lr)
        self.register("log_A_real", log_A_real, lr)
        self.register("A_imag", A_imag, lr)

        self.C = nn.Parameter(torch.view_as_real(C_init))
        self.D = nn.Parameter(torch.randn(self.h))

    def register(self, name, tensor, lr=None):
        if lr == 0.0:
            self.register_buffer(name, tensor)
        else:
            self.register_parameter(name, nn.Parameter(tensor))
            optim = {"weight_decay": 0.0}
            if lr is not None:
                optim["lr"] = lr
            setattr(getattr(self, name), "_optim", optim)

    def forward(self, u):
        if not self.transposed:
            u = u.transpose(-1, -2)
        L = u.size(-1)

        dt = torch.exp(self.log_dt)
        C = torch.view_as_complex(self.C)
        A = -torch.exp(self.log_A_real) + 1j * self.A_imag

        dtA = A * dt.unsqueeze(-1)
        K_exp = torch.exp(dtA.unsqueeze(-1) * torch.arange(L, device=u.device))
        C_tilde = C * (torch.exp(dtA) - 1.) / A
        k = 2 * torch.einsum('hn, hnl -> hl', C_tilde, K_exp).real

        k_f = torch.fft.rfft(k, n=2 * L)
        u_f = torch.fft.rfft(u, n=2 * L)
        y = torch.fft.irfft(u_f * k_f, n=2 * L)[..., :L]
        y = y + u * self.D.unsqueeze(-1)

        if not self.transposed:
            y = y.transpose(-1, -2)
        return y, None


class ConvPatchStem(nn.Module):
    """Convolutional stem for local neighborhood mixing before patch projection."""
    def __init__(self, in_channels, d_model, patch_size):
        super().__init__()
        mid_channels = max(in_channels * 8, 32)
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, stride=1, padding=1),
            nn.GELU(),
            nn.Conv2d(mid_channels, d_model, kernel_size=patch_size, stride=patch_size),
        )

    def forward(self, x):
        return self.net(x)


class MainStudyGalaxyClassifier(nn.Module):
    """Production S4D Classifier."""
    def __init__(self, s4_state=64, d_model=64, num_classes=4, colored=True,
                 num_layers=2, patch_size=1, pooling="last",
                 use_norm=False, use_residual=False, dropout=0.0,
                 patch_embed="linear"):
        super().__init__()
        self.hilbert_channels = 1 if not colored else 3
        self.patch_size = patch_size
        self.pooling = pooling
        self.use_norm = use_norm
        self.use_residual = use_residual
        self.patch_embed = patch_embed

        if patch_embed == "linear":
            self.hilbert_scan = HilbertScan(image_size=64, patch_size=patch_size)
            patch_dim = self.hilbert_channels * patch_size * patch_size
            self.uproject = nn.Linear(patch_dim, d_model)
            self.conv_stem = None
        elif patch_embed == "conv":
            self.conv_stem = ConvPatchStem(self.hilbert_channels, d_model, patch_size)
            self.hilbert_scan = HilbertScan(image_size=64 // patch_size, patch_size=1)
            self.uproject = nn.Identity()

        self.s4_layers = nn.ModuleList([
            S4DConv(d_model=d_model, d_state=s4_state, transposed=False)
            for _ in range(num_layers)
        ])
        self.acts = nn.ModuleList([nn.GELU() for _ in range(num_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(num_layers)]) if use_norm else None
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        if pooling == "last":
            self.take_last = TakeLastTimestep()
        elif pooling == "mean":
            self.take_last = None
        else:
            raise ValueError(f"Unknown pooling type {pooling}")

        self.fc = nn.Linear(d_model, num_classes)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, return_logits=True):
        if self.patch_embed == "conv":
            feat = self.conv_stem(x)
            x_seq = self.hilbert_scan(feat)
            h = self.uproject(x_seq)
        else:
            x_seq = self.hilbert_scan(x)
            h = self.uproject(x_seq)

        for i, (s4_layer, act) in enumerate(zip(self.s4_layers, self.acts)):
            residual = h
            h_in = self.norms[i](h) if self.use_norm else h
            h_out, _ = s4_layer(h_in)
            h_out = act(h_out)
            h_out = self.drop(h_out)
            h = residual + h_out if self.use_residual else h_out

        pooled = h.mean(dim=1) if self.take_last is None else self.take_last(h)
        logits = self.fc(pooled)

        if return_logits:
            return logits
        return self.softmax(logits)

# S4D Future-Work Validation — Resumable / GPU-Budgeted Kaggle Run

This version is designed specifically to survive Kaggle session/quota boundaries.

**Scope:** Future-work item 1 (richer-stem 13-cell grid), item 3 (repeat-seed noise sweep), and item 4 (GroupNorm portability/folding). Item 2 is intentionally skipped because the previous run completed its production repeat-seed comparisons.

**Important:** this notebook checkpoints **every completed epoch**, including model, optimizer, scheduler, best-validation state, history, and RNG state. If Kaggle stops the session, the next run resumes the interrupted model instead of restarting it.

**GPU accounting:** if you have 2 GPUs and Kaggle reports a remaining budget in *GPU-hours*, two simultaneous GPUs consume roughly two GPU-hours per wall-clock hour. Therefore `GPU_BUDGET_HOURS` below is converted to a conservative wall-clock deadline by dividing by the number of GPUs actually used.


In [20]:
# ===========================
# 1. Runtime / budget configuration
# ===========================
import os, json, math, time, random, copy, shutil, multiprocessing as mp
from pathlib import Path

GPU_BUDGET_HOURS = 7.0       # CHANGE THIS to your actual remaining weekly GPU quota.
USE_DUAL_GPU = True
GPU_SAFETY_MARGIN_MIN = 20   # leave time for final analysis / clean shutdown
CHECKPOINT_EVERY_EPOCH = True

# Kaggle's attached notebook-output files appear read-only under /kaggle/input.
# We copy them into /kaggle/working so this notebook can update/resume them.
BASE_DIR = Path('/kaggle/working/s4d_future_work')
RESULTS_DIR = BASE_DIR / 'results'
WEIGHTS_DIR = BASE_DIR / 'weights'
CHECKPOINT_DIR = BASE_DIR / 'checkpoints'
PLOTS_DIR = BASE_DIR / 'plots'
EXPORT_DIR = BASE_DIR / 'exports'
for d in [RESULTS_DIR, WEIGHTS_DIR, CHECKPOINT_DIR, PLOTS_DIR, EXPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Working output:', BASE_DIR)
print('CUDA available:', torch.cuda.is_available())
print('Visible GPUs:', torch.cuda.device_count())

N_GPUS = min(2, torch.cuda.device_count()) if USE_DUAL_GPU else min(1, torch.cuda.device_count())
if N_GPUS == 0:
    raise RuntimeError('No CUDA GPU is visible. In Kaggle, select Accelerator -> GPU T4 x2 (or another CUDA GPU).')

# GPU-hours are the scarce resource. With 2 GPUs, 7 GPU-hours is about 3.5 wall-hours.
WALL_BUDGET_SEC = GPU_BUDGET_HOURS * 3600 / N_GPUS
WALL_DEADLINE_SEC = max(60, WALL_BUDGET_SEC - GPU_SAFETY_MARGIN_MIN * 60)
RUN_START_TS = None
HARD_DEADLINE_TS = None

print(f'Using {N_GPUS} GPU(s).')
print(f'GPU budget: {GPU_BUDGET_HOURS:.2f} GPU-hours -> conservative wall budget: {WALL_BUDGET_SEC/3600:.2f} h')
print(f'Workers will receive a ~{WALL_DEADLINE_SEC/3600:.2f} h wall-clock training budget, leaving {GPU_SAFETY_MARGIN_MIN} min of the configured GPU budget unused.')


Working output: /kaggle/working/s4d_future_work
CUDA available: True
Visible GPUs: 2
Using 2 GPU(s).
GPU budget: 7.00 GPU-hours -> conservative wall budget: 3.50 h
Workers will receive a ~3.17 h wall-clock training budget, leaving 20 min of the configured GPU budget unused.


## 2. Import the previous timed-out run

Before running this notebook, use **Add Data → Notebook Output Files → Your Work** and attach the previous notebook's output. Kaggle mounts attached files under `/kaggle/input/`. The cell below searches all attached notebook-output trees for `results/`, `weights/`, and `checkpoints/` and copies them into this notebook's writable `/kaggle/working/s4d_future_work/`. Kaggle supports attaching a notebook's output directly to another notebook. urlKaggle notebook output discussionturn0search2

If the old run only saved final `.pt` files, those completed runs are recoverable. The old notebook did **not** save mid-epoch checkpoints, so a model killed before its final save cannot be resumed from the old run; this notebook fixes that going forward.


In [21]:
# ===========================
# 3. Import previous output / cache
# ===========================
from pathlib import Path

def copy_tree_files(src_root, dst_root):
    copied = 0
    if not src_root.exists():
        return copied
    for src in src_root.rglob('*'):
        if src.is_file():
            rel = src.relative_to(src_root)
            dst = dst_root / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            # Never overwrite a newer/current result.
            if not dst.exists():
                shutil.copy2(src, dst)
                copied += 1
    return copied

attached = Path('/kaggle/input')
found = []
if attached.exists():
    for p in attached.rglob('*'):
        if p.is_dir() and p.name in {'s4d_future_work', 'results', 'weights', 'checkpoints'}:
            found.append(p)

# Prefer an attached top-level s4d_future_work directory. Otherwise find individual folders.
source_roots = []
for p in found:
    if p.name == 's4d_future_work':
        source_roots.append((p, BASE_DIR))

if not source_roots:
    for p in found:
        if p.name in {'results', 'weights', 'checkpoints'}:
            source_roots.append((p, BASE_DIR / p.name))

counts = 0
for src, dst in source_roots:
    counts += copy_tree_files(src, dst)

print(f'Imported {counts} file(s) from attached Kaggle notebook outputs.')
print('Existing final weights:', len(list(WEIGHTS_DIR.glob('*.pt'))))
print('Existing results:', len(list(RESULTS_DIR.glob('*.json'))))
print('Existing resumable checkpoints:', len(list(CHECKPOINT_DIR.glob('*.pt'))))

# Also show exactly what Kaggle attached, which is useful if the old output is not found.
if attached.exists():
    print('\n/kaggle/input top level:')
    for p in sorted(attached.iterdir()):
        print(' ', p)


Imported 39 file(s) from attached Kaggle notebook outputs.
Existing final weights: 13
Existing results: 13
Existing resumable checkpoints: 0

/kaggle/input top level:
  /kaggle/input/notebooks


In [22]:
# ===========================
# 4. Reproducibility + data
# ===========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from model.functions import load_data

CLASS_NAMES = ['Smooth Round', 'Smooth Cigar', 'Edge-on Disk', 'Unbarred Spiral']
SPLIT_SEED = 30485
MAIN_EPOCHS = 630
BATCH_SIZE = 32

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def make_augmented_dataset(X, y):
    class AugmentedGalaxyDataset(Dataset):
        def __init__(self, X, y): self.X, self.y = X, y
        def __len__(self): return len(self.X)
        def __getitem__(self, idx):
            img, label = self.X[idx], self.y[idx]
            k = random.randint(0, 3)
            if k: img = torch.rot90(img, k, dims=(1, 2))
            if random.random() < 0.5: img = torch.flip(img, dims=(2,))
            if random.random() < 0.5: img = torch.flip(img, dims=(1,))
            return img, label
    return AugmentedGalaxyDataset(X, y)

# Keep the same split and dataset recipe as the report/notebook.
X, y_onehot, y = load_data(root='./data', download=True, train=True, colored=True)
X_test, y_test_onehot, y_test = load_data(root='./data', download=True, train=False, colored=True)
x_train, x_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SPLIT_SEED, stratify=y
)
DATA_SPLIT = {'train': (x_train, y_train), 'val': (x_val, y_val), 'test': (X_test, y_test)}
print('RGB train/val/test:', x_train.shape, x_val.shape, X_test.shape)


100%|██████████| 68.7M/68.7M [00:17<00:00, 3.93MB/s]
100%|██████████| 17.3M/17.3M [00:01<00:00, 10.4MB/s]


Original Dataset Size: 8000 samples
Original Dataset Size: 2000 samples
RGB train/val/test: torch.Size([6400, 3, 64, 64]) torch.Size([1600, 3, 64, 64]) torch.Size([2000, 3, 64, 64])


In [23]:
# ===========================
# 5. Richer-stem grid reconstruction
# ===========================
class RicherStem(nn.Module):
    def __init__(self, depth, in_channels=3, d_model=64, mid_channels=32, dropout=0.1):
        super().__init__()
        if depth not in (1,2,3,4): raise ValueError(depth)
        self.depth = depth
        self.grid = 64 if depth == 1 else 16
        def gn(ch): return nn.GroupNorm(8 if ch % 8 == 0 else 1, ch)
        if depth == 1:
            self.conv1=nn.Conv2d(in_channels,d_model,3,1,1); self.norm1=gn(d_model); self.act1=nn.GELU()
        elif depth == 2:
            self.conv1=nn.Conv2d(in_channels,mid_channels,3,1,1); self.norm1=gn(mid_channels); self.act1=nn.GELU()
            self.conv2=nn.Conv2d(mid_channels,d_model,3,4,1); self.norm2=gn(d_model); self.act2=nn.GELU()
        elif depth == 3:
            self.conv1=nn.Conv2d(in_channels,mid_channels,3,1,1); self.norm1=gn(mid_channels); self.act1=nn.GELU()
            self.conv2=nn.Conv2d(mid_channels,mid_channels,3,2,1); self.norm2=gn(mid_channels); self.act2=nn.GELU()
            self.conv3=nn.Conv2d(mid_channels,d_model,3,2,1); self.norm3=gn(d_model); self.act3=nn.GELU()
        else:
            self.conv1=nn.Conv2d(in_channels,mid_channels,3,1,1); self.norm1=gn(mid_channels); self.act1=nn.GELU()
            self.res_conv=nn.Conv2d(mid_channels,mid_channels,3,1,1); self.res_norm=gn(mid_channels); self.res_act=nn.GELU(); self.drop=nn.Dropout2d(dropout)
            self.conv2=nn.Conv2d(mid_channels,mid_channels,3,2,1); self.norm2=gn(mid_channels); self.act2=nn.GELU()
            self.conv3=nn.Conv2d(mid_channels,d_model,3,2,1); self.norm3=gn(d_model); self.act3=nn.GELU()
    def forward(self,x):
        if self.depth==1: return self.act1(self.norm1(self.conv1(x)))
        if self.depth==2:
            x=self.act1(self.norm1(self.conv1(x))); return self.act2(self.norm2(self.conv2(x)))
        if self.depth==3:
            x=self.act1(self.norm1(self.conv1(x))); x=self.act2(self.norm2(self.conv2(x))); return self.act3(self.norm3(self.conv3(x)))
        x=self.act1(self.norm1(self.conv1(x))); r=self.res_act(self.res_norm(self.res_conv(x))); x=self.drop(x+r); x=self.act2(self.norm2(self.conv2(x))); return self.act3(self.norm3(self.conv3(x)))

class RicherGridModel(nn.Module):
    def __init__(self, stem_depth, num_s4_layers, d_model=64, s4_state=64, num_classes=4, stem_dropout=0.1):
        super().__init__()
        self.stem_depth=stem_depth; self.num_s4_layers=num_s4_layers
        self.cnn_stem=RicherStem(stem_depth,d_model=d_model,dropout=stem_dropout)
        self.grid=self.cnn_stem.grid; self.seq_len=self.grid*self.grid
        from model.hilbert import HilbertScan
        self.hilbert_scan=HilbertScan(n=self.grid)
        self.s4_layers=nn.ModuleList([S4D(d_model=d_model,d_state=s4_state,transposed=False) for _ in range(num_s4_layers)])
        self.acts=nn.ModuleList([nn.GELU() for _ in range(num_s4_layers)])
        self.take_last=TakeLastTimestep(); self.fc=nn.Linear(d_model,num_classes)
    def forward(self,x,return_logits=True):
        h=self.hilbert_scan(self.cnn_stem(x))
        for layer,act in zip(self.s4_layers,self.acts):
            h,_=layer(h); h=act(h)
        logits=self.fc(self.take_last(h))
        return logits if return_logits else torch.softmax(logits,dim=-1)

def make_richer_grid_model(depth,s4_layers): return RicherGridModel(depth,s4_layers)

RICHER_REPORT_PARAMS={(1,0):2180,(1,1):10500,(1,2):18820,(2,0):19844,(2,1):28164,(2,2):36484,(3,0):29156,(3,1):37476,(3,2):45796,(4,0):38468,(4,1):46788,(4,2):55108}
RICHER_REPORT_SEQ={1:4096,2:256,3:256,4:256}

checks=[]
for d in (1,2,3,4):
    for n in (0,1,2):
        m=make_richer_grid_model(d,n); p=sum(x.numel() for x in m.parameters())
        checks.append({'depth':d,'s4_layers':n,'params':p,'expected':RICHER_REPORT_PARAMS[(d,n)],'seq_len':m.seq_len,'expected_seq':RICHER_REPORT_SEQ[d],'ok':p==RICHER_REPORT_PARAMS[(d,n)] and m.seq_len==RICHER_REPORT_SEQ[d]})
raw=GalaxyClassifierS4D(s4_state=64,d_model=64,num_classes=4,colored=True)
checks.append({'depth':0,'s4_layers':2,'params':sum(x.numel() for x in raw.parameters()),'expected':17156,'seq_len':4096,'expected_seq':4096,'ok':sum(x.numel() for x in raw.parameters())==17156})
check_df=pd.DataFrame(checks); display(check_df)
if not check_df.ok.all(): raise RuntimeError('Architecture reconstruction failed report parameter/sequence checks.')


,depth,s4_layers,params,expected,seq_len,expected_seq,ok
0,1,0,2180,2180,4096,4096,True
1,1,1,10500,10500,4096,4096,True
2,1,2,18820,18820,4096,4096,True
3,2,0,19844,19844,256,256,True
4,2,1,28164,28164,256,256,True
5,2,2,36484,36484,256,256,True
6,3,0,29156,29156,256,256,True
7,3,1,37476,37476,256,256,True
8,3,2,45796,45796,256,256,True
9,4,0,38468,38468,256,256,True


In [24]:
# ===========================
# 6. Specs + checkpoint-resumable training engine
# ===========================
RICHER_SPECS=[]
for d in (1,2,3,4):
    for n in (0,1,2):
        RICHER_SPECS.append({'id':f'richer_d{d}_s4{n}','label':f'Richer stem depth {d} + {n} S4D','builder':lambda d=d,n=n:make_richer_grid_model(d,n),'expected_params':RICHER_REPORT_PARAMS[(d,n)],'depth':d,'s4_layers':n})
RAW_SPEC={'id':'richer_rawpix_s4d2','label':'Richer raw-pixel S4D-only baseline','builder':lambda:GalaxyClassifierS4D(s4_state=64,d_model=64,num_classes=4,colored=True),'expected_params':17156,'depth':0,'s4_layers':2}
RICHER_ALL_SPECS=RICHER_SPECS+[RAW_SPEC]
SPEC_BY_ID={s['id']:s for s in RICHER_ALL_SPECS}

MAIN_RECIPE={'batch_size':32,'lr':1e-3,'weight_decay':1e-2,'grad_clip':1.0,'epochs':630}

def run_id(slug,seed): return f'{slug}__main__seed{seed}'
def checkpoint_path(rid): return CHECKPOINT_DIR/f'{rid}.pt'
def result_path(rid): return RESULTS_DIR/f'{rid}.json'
def weights_path(rid): return WEIGHTS_DIR/f'{rid}.pt'

def make_optimizer(model):
    decay,no_decay,special=[],[],[]
    for name,p in model.named_parameters():
        if not p.requires_grad: continue
        if hasattr(p,'_optim'):
            special.append({'params':[p],'lr':getattr(p,'_optim',{}).get('lr',1e-3),'weight_decay':0.0})
        elif any(k in name.lower() for k in ['bias','norm','layernorm']): no_decay.append(p)
        else: decay.append(p)
    return torch.optim.AdamW([{'params':decay,'lr':1e-3,'weight_decay':1e-2},{'params':no_decay,'lr':1e-3,'weight_decay':0.0}]+special)

def make_scheduler(opt): return torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt,T_0=10,T_mult=2,eta_min=1e-5)

def metrics(y_true,y_pred,probs):
    return {'accuracy':float(accuracy_score(y_true,y_pred)),'f1_macro':float(f1_score(y_true,y_pred,average='macro')),'precision_macro':float(precision_score(y_true,y_pred,average='macro',zero_division=0)),'recall_macro':float(recall_score(y_true,y_pred,average='macro',zero_division=0)),'roc_auc_macro':float(roc_auc_score(y_true,probs,multi_class='ovr',average='macro'))}

def evaluate(model,loader,device):
    model.eval(); ys=[]; ps=[]; probs=[]
    with torch.no_grad():
        for x,y in loader:
            x=x.to(device); logits=model(x,return_logits=True); p=torch.softmax(logits,1)
            ys.extend(y.numpy().tolist()); ps.extend(logits.argmax(1).cpu().numpy().tolist()); probs.extend(p.cpu().numpy().tolist())
    out=metrics(np.array(ys),np.array(ps),np.array(probs)); out['confusion_matrix']=confusion_matrix(ys,ps).tolist(); return out

def loaders(seed):
    set_all_seeds(seed); tx,ty=DATA_SPLIT['train']; vx,vy=DATA_SPLIT['val']; qx,qy=DATA_SPLIT['test']
    return (DataLoader(make_augmented_dataset(tx,ty),batch_size=32,shuffle=True),DataLoader(TensorDataset(vx,vy),batch_size=32,shuffle=False),DataLoader(TensorDataset(qx,qy),batch_size=64,shuffle=False))

def _rng_state():
    s={'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state()}
    if torch.cuda.is_available(): s['cuda']=torch.cuda.get_rng_state_all()
    return s

def _restore_rng(s):
    random.setstate(s['python']); np.random.set_state(s['numpy']); torch.set_rng_state(s['torch'])
    if torch.cuda.is_available() and 'cuda' in s: torch.cuda.set_rng_state_all(s['cuda'])

def save_checkpoint(rid,epoch,model,opt,sched,best_state,best_val,history,seed):
    state={'epoch':epoch,'model_state':model.state_dict(),'optimizer_state':opt.state_dict(),'scheduler_state':sched.state_dict(),'best_state':best_state,'best_val':best_val,'history':history,'seed':seed,'rng_state':_rng_state()}
    tmp=checkpoint_path(rid).with_suffix('.tmp')
    torch.save(state,tmp); os.replace(tmp,checkpoint_path(rid))

def train_model_resumable(spec,seed,purpose,device,deadline_ts):
    rid=run_id(spec['id'],seed); rp=result_path(rid); wp=weights_path(rid); cp=checkpoint_path(rid)
    if rp.exists() and wp.exists():
        r=json.loads(rp.read_text()); r['_cached']=True; print(f'[{rid}] final result already exists; skipping.'); return r
    set_all_seeds(seed)
    train_loader,val_loader,test_loader=loaders(seed)
    model=spec['builder']().to(device); actual=sum(p.numel() for p in model.parameters())
    if actual!=spec['expected_params']: raise RuntimeError(f'{rid}: expected {spec["expected_params"]}, got {actual}')
    opt=make_optimizer(model); sched=make_scheduler(opt); loss_fn=nn.CrossEntropyLoss()
    epochs=MAIN_RECIPE['epochs']; history={'train_loss':[],'train_acc':[],'val_acc':[],'lr':[]}; best_val=-1.; best_state=None; start_epoch=0
    if cp.exists():
        print(f'[{rid}] RESUMING from checkpoint {cp}')
        ck=torch.load(cp,map_location='cpu',weights_only=False)
        model.load_state_dict(ck['model_state']); opt.load_state_dict(ck['optimizer_state']); sched.load_state_dict(ck['scheduler_state'])
        best_state=ck['best_state']; best_val=ck['best_val']; history=ck['history']; start_epoch=ck['epoch']; _restore_rng(ck['rng_state'])
        model.to(device)
    t0=time.time()
    for epoch in range(start_epoch,epochs):
        if time.time() >= deadline_ts:
            print(f'[{rid}] deadline reached before epoch {epoch+1}; checkpoint at epoch {epoch} retained.')
            return {'run_id':rid,'architecture':spec['id'],'seed':seed,'status':'paused','epoch':epoch}
        model.train(); total=correct=0; running=0.0
        for x,y in train_loader:
            x=x.to(device); y=y.to(device); opt.zero_grad(set_to_none=True); logits=model(x,return_logits=True); loss=loss_fn(logits,y); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
            running += float(loss.item())*y.size(0); correct += int((logits.argmax(1)==y).sum()); total += y.size(0)
        sched.step(); val=evaluate(model,val_loader,device)
        history['train_loss'].append(running/max(1,total)); history['train_acc'].append(correct/max(1,total)); history['val_acc'].append(val['accuracy']); history['lr'].append(float(opt.param_groups[0]['lr']))
        if val['accuracy']>best_val:
            best_val=val['accuracy']; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        save_checkpoint(rid,epoch+1,model,opt,sched,best_state,best_val,history,seed)
        if (epoch+1)%25==0 or epoch==epochs-1: print(f'[{rid}] epoch {epoch+1}/{epochs} train={history["train_acc"][-1]:.4f} val={val["accuracy"]:.4f}')
    model.load_state_dict(best_state); test=evaluate(model,test_loader,device); elapsed=time.time()-t0
    torch.save(model.state_dict(),wp)
    result={'run_id':rid,'purpose':purpose,'architecture':spec['id'],'label':spec['label'],'seed':seed,'params':actual,'expected_params':spec['expected_params'],'epochs':epochs,'train_time_sec':elapsed,**{k:v for k,v in test.items() if k!='confusion_matrix'},'confusion_matrix':test['confusion_matrix'],'history':history,'weights_file':str(wp),'recipe':'main','split_seed':SPLIT_SEED,'status':'complete'}
    rp.write_text(json.dumps(result,indent=2));
    # The final result is now authoritative; the resumable checkpoint can be removed to reduce clutter.
    try: cp.unlink()
    except FileNotFoundError: pass
    print(f'[{rid}] COMPLETE in {elapsed/60:.1f} min | test={test["accuracy"]:.4f}')
    return result


## 7. Priority queue

The queue deliberately starts with `richer_d4_s42` because it is required for the GroupNorm portability experiment. The old completed `richer_d1_s40` is reused if its final result/weights are imported. Item 2 is not scheduled.


In [25]:
# ===========================
# 7. Build priority queue
# ===========================
MAIN_SEED=30485
NOISE_SEED=8842

# Item 1: d4/s4=2 first, then all remaining main-grid configurations.
item1=[{'spec_id':s['id'],'seed':MAIN_SEED,'purpose':'richer_grid_main_recipe'} for s in RICHER_ALL_SPECS]
item1.sort(key=lambda j: (0 if j['spec_id']=='richer_d4_s42' else 1, j['spec_id']))
# Item 3 comes after item 1.
item3=[{'spec_id':s['id'],'seed':NOISE_SEED,'purpose':'richer_grid_noise_repeat_seed'} for s in RICHER_ALL_SPECS]

def is_complete(j): return result_path(run_id(j['spec_id'],j['seed'])).exists() and weights_path(run_id(j['spec_id'],j['seed'])).exists()

queue=[j for j in item1+item3 if not is_complete(j)]
print(f'Candidate training runs: {len(item1)+len(item3)} | already complete: {len(item1)+len(item3)-len(queue)} | remaining: {len(queue)}')
for j in queue: print(' ',j['purpose'],j['spec_id'],j['seed'])


Candidate training runs: 26 | already complete: 1 | remaining: 25
  richer_grid_main_recipe richer_d4_s42 30485
  richer_grid_main_recipe richer_d1_s41 30485
  richer_grid_main_recipe richer_d1_s42 30485
  richer_grid_main_recipe richer_d2_s40 30485
  richer_grid_main_recipe richer_d2_s41 30485
  richer_grid_main_recipe richer_d2_s42 30485
  richer_grid_main_recipe richer_d3_s40 30485
  richer_grid_main_recipe richer_d3_s41 30485
  richer_grid_main_recipe richer_d3_s42 30485
  richer_grid_main_recipe richer_d4_s40 30485
  richer_grid_main_recipe richer_d4_s41 30485
  richer_grid_main_recipe richer_rawpix_s4d2 30485
  richer_grid_noise_repeat_seed richer_d1_s40 8842
  richer_grid_noise_repeat_seed richer_d1_s41 8842
  richer_grid_noise_repeat_seed richer_d1_s42 8842
  richer_grid_noise_repeat_seed richer_d2_s40 8842
  richer_grid_noise_repeat_seed richer_d2_s41 8842
  richer_grid_noise_repeat_seed richer_d2_s42 8842
  richer_grid_noise_repeat_seed richer_d3_s40 8842
  richer_grid_noise_

## 8. Dual-GPU resumable scheduler

This uses separate **processes**, not threads. Each worker owns one CUDA context and has its own RNG state. The workers share only the filesystem checkpoint/result cache. The deadline is based on total GPU-hours, so two GPUs do not accidentally consume a 7-GPU-hour budget in 7 wall-clock hours.


In [26]:
# ===========================
# 8. Separate-process dual-GPU workers
# ===========================
# IMPORTANT: use fork only after all model/data definitions are loaded but before the parent creates a CUDA context.
# The parent deliberately never moves a model to CUDA. Each child initializes its own GPU.

def worker_loop(gpu_id, jobs, deadline_ts, result_queue):
    try:
        device=f'cuda:{gpu_id}'
        torch.cuda.set_device(gpu_id)
        results=[]
        for job in jobs:
            if time.time() >= deadline_ts: break
            spec=SPEC_BY_ID[job['spec_id']]
            try:
                r=train_model_resumable(spec,job['seed'],job['purpose'],device,deadline_ts); results.append(r)
            except Exception as exc:
                print(f'[GPU {gpu_id}] ERROR in {job["spec_id"]}: {type(exc).__name__}: {exc}')
                results.append({'run_id':run_id(job['spec_id'],job['seed']),'status':'error','error':repr(exc)})
        result_queue.put({'gpu':gpu_id,'results':results})
    except Exception as exc:
        result_queue.put({'gpu':gpu_id,'fatal':repr(exc)})

if queue:
    # Start the GPU-budget clock only now: dataset download / CPU setup should not
    # consume the training wall-clock budget.
    RUN_START_TS = time.time()
    HARD_DEADLINE_TS = RUN_START_TS + WALL_DEADLINE_SEC
    print(f'GPU training clock started. Hard deadline in {WALL_DEADLINE_SEC/3600:.2f} h.')
    # Round-robin assignment. Jobs are intentionally small enough that the two queues should stay reasonably balanced.
    job_lists=[queue[g::N_GPUS] for g in range(N_GPUS)]
    ctx=mp.get_context('fork')
    result_queue=ctx.Queue()
    procs=[]
    for g,jobs in enumerate(job_lists):
        p=ctx.Process(target=worker_loop,args=(g,jobs,HARD_DEADLINE_TS,result_queue),daemon=False)
        p.start(); procs.append(p)
        print(f'Launched GPU {g}: pid={p.pid}, {len(jobs)} queued jobs, first={jobs[0]["spec_id"] if jobs else "none"}')
    while any(p.is_alive() for p in procs):
        time.sleep(30)
        done=sum(1 for j in queue if is_complete(j))
        remaining=max(0,HARD_DEADLINE_TS-time.time())
        print(f'[monitor] complete={done}/{len(queue)} | wall time remaining={remaining/60:.1f} min')
        if time.time() >= HARD_DEADLINE_TS:
            print('[monitor] deadline reached; terminating workers. Completed checkpoints remain resumable.')
            for p in procs:
                if p.is_alive(): p.terminate()
            break
    for p in procs: p.join(timeout=20)
    print('Worker exit codes:',[p.exitcode for p in procs])
else:
    print('No training required: all queued runs already have final results + weights.')


GPU training clock started. Hard deadline in 3.17 h.
Launched GPU 0: pid=107, 13 queued jobs, first=richer_d4_s42
Launched GPU 1: pid=109, 12 queued jobs, first=richer_d1_s41
[monitor] complete=0/25 | wall time remaining=189.5 min
Worker exit codes: [0, 0]


## 9. Main-grid results

This cell is safe to run even if the training phase stopped early. It reports only completed final results.


In [27]:
# ===========================
# 9. Main-grid result table
# ===========================
def completed_result(spec,seed):
    p=result_path(run_id(spec['id'],seed))
    return json.loads(p.read_text()) if p.exists() else None

rows=[]
for s in RICHER_ALL_SPECS:
    r=completed_result(s,MAIN_SEED)
    if r:
        rows.append({k:r.get(k) for k in ['run_id','architecture','label','seed','params','accuracy','f1_macro','precision_macro','recall_macro','roc_auc_macro','train_time_sec']})
richer_main_df=pd.DataFrame(rows)
if len(richer_main_df):
    richer_main_df['stem_depth']=[SPEC_BY_ID[a].get('depth') for a in richer_main_df.architecture]
    richer_main_df['s4_layers']=[SPEC_BY_ID[a].get('s4_layers') for a in richer_main_df.architecture]
    display(richer_main_df.sort_values(['stem_depth','s4_layers']))
    richer_main_df.to_csv(RESULTS_DIR/'richer_grid_main_results.csv',index=False)
else: print('No completed main-grid runs yet.')


,run_id,architecture,label,seed,params,accuracy,f1_macro,precision_macro,recall_macro,roc_auc_macro,train_time_sec,stem_depth,s4_layers
0,richer_d1_s40__main__seed30485,richer_d1_s40,Richer stem depth 1 + 0 S4D,30485,2180,0.597,0.59576,0.609367,0.59829,0.843252,2427.893655,1,0


## 10. Noise robustness repeat

Only completed seed-8842 models are evaluated. Missing models remain in the queue for the next resumed session.


In [28]:
# ===========================
# 10. Noise sweep
# ===========================
NOISE_SIGMAS=[0.0,0.05,0.10,0.20,0.30]
NOISE_CLIP_TO_UNIT=True

def evaluate_with_noise(model,sigma,device,batch_size=64):
    model.eval(); ds=TensorDataset(DATA_SPLIT['test'][0],DATA_SPLIT['test'][1]); loader=DataLoader(ds,batch_size=batch_size,shuffle=False)
    ys=[]; ps=[]; probs=[]
    with torch.no_grad():
        for x,y in loader:
            x=x.to(device); noisy=(x+sigma*torch.randn_like(x)).clamp(0,1) if NOISE_CLIP_TO_UNIT else x+sigma*torch.randn_like(x)
            logits=model(noisy,return_logits=True); p=torch.softmax(logits,1)
            ys.extend(y.numpy().tolist()); ps.extend(logits.argmax(1).cpu().numpy().tolist()); probs.extend(p.cpu().numpy().tolist())
    return metrics(np.array(ys),np.array(ps),np.array(probs))

def noise_eval_all():
    device='cuda:0' if torch.cuda.is_available() else 'cpu'; rows=[]
    for s in RICHER_ALL_SPECS:
        r=completed_result(s,NOISE_SEED)
        if not r: continue
        m=s['builder']().to(device); m.load_state_dict(torch.load(weights_path(run_id(s['id'],NOISE_SEED)),map_location=device,weights_only=True))
        for sigma in NOISE_SIGMAS:
            mm=evaluate_with_noise(m,sigma,device)
            rows.append({'seed':NOISE_SEED,'architecture':s['id'],'label':s['label'],'stem_depth':s.get('depth'),'s4_layers':s.get('s4_layers'),'sigma':sigma,'accuracy':mm['accuracy'],'f1_macro':mm['f1_macro'],'precision_macro':mm['precision_macro'],'recall_macro':mm['recall_macro'],'roc_auc_macro':mm['roc_auc_macro']})
        del m
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    return pd.DataFrame(rows)

noise_repeat_eval_df=noise_eval_all(); noise_repeat_eval_df.to_csv(RESULTS_DIR/'noise_repeat_seed_8842.csv',index=False)
display(noise_repeat_eval_df.head(25))


""


## 11. GroupNorm portability test

This remains the calibrated fixed-statistics approximation from the original notebook. True GroupNorm cannot be exactly folded into a convolution because its normalization statistics depend on the current input. The test therefore measures the accuracy cost of replacing dynamic GN with training-set calibrated fixed group statistics.


In [29]:
# ===========================
# 11. GroupNorm folding
# ===========================
def gn_stats(x,gn):
    B,C,H,W=x.shape; G=gn.num_groups; y=x.reshape(B,G,C//G,H,W); m=y.mean((2,3,4)); v=y.var((2,3,4),unbiased=False); return m,v

def collect_gn_calibration(model,device):
    sums={}; sums2={}; counts={}; hooks=[]
    for name,mod in model.cnn_stem.named_modules():
        if isinstance(mod,nn.GroupNorm):
            sums[name]=None; sums2[name]=None; counts[name]=0
            def hook(mod,inp,out,name=name):
                m,v=gn_stats(inp[0].detach(),mod); sm=m.sum(0); sm2=(v+m*m).sum(0)
                if sums[name] is None: sums[name]=sm.clone(); sums2[name]=sm2.clone()
                else: sums[name]+=sm; sums2[name]+=sm2
                counts[name]+=m.shape[0]
            hooks.append(mod.register_forward_hook(hook))
    loader=DataLoader(TensorDataset(DATA_SPLIT['train'][0],DATA_SPLIT['train'][1]),batch_size=64,shuffle=False)
    model.eval()
    with torch.no_grad():
        for x,_ in loader: model(x.to(device),return_logits=True)
    for h in hooks: h.remove()
    out={}
    for name in sums:
        mean=sums[name]/counts[name]; second=sums2[name]/counts[name]; out[name]={'mean':mean.cpu(),'var':(second-mean*mean).clamp_min(0).cpu(),'count':counts[name]}
    return out

def fold_conv_gn(conv,gn,mean_g,var_g):
    C=conv.out_channels; G=gn.num_groups; gs=C//G; dev=conv.weight.device
    mean=mean_g.to(dev).repeat_interleave(gs); var=var_g.to(dev).repeat_interleave(gs)
    scale=gn.weight.to(dev)/torch.sqrt(var+gn.eps); shift=gn.bias.to(dev)-scale*mean
    with torch.no_grad():
        conv.weight.mul_(scale.view(-1,1,1,1))
        if conv.bias is None: conv.bias=nn.Parameter(shift.clone())
        else: conv.bias.mul_(scale).add_(shift)

def fold_richer_d4(model,stats):
    stem=model.cnn_stem
    for cn,gnn in [('conv1','norm1'),('res_conv','res_norm'),('conv2','norm2'),('conv3','norm3')]:
        if hasattr(stem,cn) and hasattr(stem,gnn):
            conv=getattr(stem,cn); gn=getattr(stem,gnn); fold_conv_gn(conv,gn,stats[gnn]['mean'],stats[gnn]['var']); setattr(stem,gnn,nn.Identity())
    return model

fold_id=run_id('richer_d4_s42',MAIN_SEED); fold_weight=weights_path(fold_id)
if fold_weight.exists():
    device='cuda:0' if torch.cuda.is_available() else 'cpu'; spec=SPEC_BY_ID['richer_d4_s42']
    original=spec['builder']().to(device); original.load_state_dict(torch.load(fold_weight,map_location=device,weights_only=True)); original.eval()
    stats=collect_gn_calibration(original,device); folded=copy.deepcopy(original); fold_richer_d4(folded,stats); folded.eval()
    vl=DataLoader(TensorDataset(DATA_SPLIT['val'][0],DATA_SPLIT['val'][1]),batch_size=64); tl=DataLoader(TensorDataset(DATA_SPLIT['test'][0],DATA_SPLIT['test'][1]),batch_size=64)
    ov=evaluate(original,vl,device); fv=evaluate(folded,vl,device); ot=evaluate(original,tl,device); ft=evaluate(folded,tl,device)
    fold_summary=pd.DataFrame([{'model':'Original GroupNorm','split':'val',**{k:v for k,v in ov.items() if k!='confusion_matrix'}},{'model':'Calibrated fixed-GN folded','split':'val',**{k:v for k,v in fv.items() if k!='confusion_matrix'}},{'model':'Original GroupNorm','split':'test',**{k:v for k,v in ot.items() if k!='confusion_matrix'}},{'model':'Calibrated fixed-GN folded','split':'test',**{k:v for k,v in ft.items() if k!='confusion_matrix'}}])
    display(fold_summary); print('Test accuracy change (pp):',(ft['accuracy']-ot['accuracy'])*100)
    torch.save(folded.state_dict(),EXPORT_DIR/'richer_d4_s42_gn_folded_state.pt')
    (EXPORT_DIR/'richer_d4_s42_gn_calibration.json').write_text(json.dumps({k:{'mean':v['mean'].tolist(),'var':v['var'].tolist(),'count':v['count']} for k,v in stats.items()},indent=2))
    fold_summary.to_csv(RESULTS_DIR/'gn_fold_summary.csv',index=False)
else:
    fold_summary=pd.DataFrame(); print('richer_d4_s42 is not complete yet; GroupNorm test will run automatically after that model finishes in a later resumed session.')


richer_d4_s42 is not complete yet; GroupNorm test will run automatically after that model finishes in a later resumed session.


## 12. Resume / completion status

Run this cell at the end of every session. It shows exactly what is complete and what will be picked up next time.


In [30]:
# ===========================
# 12. Completion manifest
# ===========================
status_rows=[]
for purpose,jobs in [('item1',item1),('item3',item3)]:
    for j in jobs:
        rid=run_id(j['spec_id'],j['seed']); status_rows.append({'purpose':purpose,'run_id':rid,'complete':is_complete(j),'checkpoint':checkpoint_path(rid).exists()})
status_df=pd.DataFrame(status_rows); display(status_df)
print(f'Complete: {status_df.complete.sum()}/{len(status_df)}')
print(f'Partial checkpoints: {status_df.checkpoint.sum()}')

manifest={'gpu_budget_hours':GPU_BUDGET_HOURS,'gpus_used':N_GPUS,'wall_budget_hours':WALL_BUDGET_SEC/3600,'main_seed':MAIN_SEED,'noise_seed':NOISE_SEED,'epochs':MAIN_EPOCHS,'completed_runs':int(status_df.complete.sum()),'total_training_runs':len(status_df),'timestamp':time.strftime('%Y-%m-%d %H:%M:%S')}
(RESULTS_DIR/'resume_manifest.json').write_text(json.dumps(manifest,indent=2))
print(json.dumps(manifest,indent=2))


,purpose,run_id,complete,checkpoint
0,item1,richer_d4_s42__main__seed30485,False,False
1,item1,richer_d1_s40__main__seed30485,True,False
2,item1,richer_d1_s41__main__seed30485,False,False
3,item1,richer_d1_s42__main__seed30485,False,False
4,item1,richer_d2_s40__main__seed30485,False,False
5,item1,richer_d2_s41__main__seed30485,False,False
6,item1,richer_d2_s42__main__seed30485,False,False
7,item1,richer_d3_s40__main__seed30485,False,False
8,item1,richer_d3_s41__main__seed30485,False,False
9,item1,richer_d3_s42__main__seed30485,False,False


Complete: 1/26
Partial checkpoints: 0
{
  "gpu_budget_hours": 7.0,
  "gpus_used": 2,
  "wall_budget_hours": 3.5,
  "main_seed": 30485,
  "noise_seed": 8842,
  "epochs": 630,
  "completed_runs": 1,
  "total_training_runs": 26,
  "timestamp": "2026-08-18 01:52:02"
}
